# Сегментация дефектов дорожного покрытия на основе СНС
Курсовая работа, Черноиван А. Н., гр. ИИ-26.

**Перед запуском:** справа в панели *Input* добавьте датасет `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`; в *Settings* включите **Accelerator: GPU** и **Internet: On**. Затем *Run All*. Полный прогон занимает ~1,5–2 ч.

## 0. Код проекта
Ячейки ниже создают файлы проекта в `/kaggle/working/road_defect_seg`.

In [ ]:
import os
os.makedirs('/kaggle/working/road_defect_seg', exist_ok=True)
%cd /kaggle/working/road_defect_seg
for d in ['src', 'scripts', 'tests', 'configs']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/__init__.py
__version__ = "1.0"


In [ ]:
%%writefile src/config.py
SOURCE_CLASSES = {0: "pothole", 1: "crack", 2: "manhole"}

CLASS_NAMES = ["background", "pothole", "crack", "manhole"]
CLASS_NAMES_RU = ["фон", "выбоина", "трещина", "люк"]
NUM_CLASSES = len(CLASS_NAMES)
DEFECT_CLASS_IDS = [1, 2, 3]

CLASS_COLORS = {
    0: (0, 0, 0),
    1: (230, 40, 40),
    2: (255, 210, 0),
    3: (40, 120, 255),
}

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

DEFAULT_IMG_H = 352
DEFAULT_IMG_W = 640

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

SEED = 42


In [ ]:
%%writefile src/labels.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np


@dataclass
class Annotation:
    cls_id: int
    polygon: np.ndarray

    @property
    def bbox(self) -> tuple[float, float, float, float]:
        xs, ys = self.polygon[:, 0], self.polygon[:, 1]
        return float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())

    @property
    def area(self) -> float:
        x, y = self.polygon[:, 0], self.polygon[:, 1]
        return float(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))


def parse_line(line: str, img_w: int, img_h: int) -> Annotation | None:
    parts = line.strip().split()
    if not parts:
        return None
    try:
        values = [float(p) for p in parts]
    except ValueError:
        return None
    cls_id = int(values[0])
    coords = values[1:]

    if len(coords) == 4:
        xc, yc, w, h = coords
        pts = np.array([
            [xc - w / 2, yc - h / 2], [xc + w / 2, yc - h / 2],
            [xc + w / 2, yc + h / 2], [xc - w / 2, yc + h / 2],
        ], dtype=np.float32)
    elif len(coords) >= 6 and len(coords) % 2 == 0:
        pts = np.asarray(coords, dtype=np.float32).reshape(-1, 2)
    else:
        return None

    pts = np.clip(pts, 0.0, 1.0)
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h
    ann = Annotation(cls_id=cls_id, polygon=pts)
    if ann.area < 1.0:
        return None
    return ann


def read_label_file(path: str | Path, img_w: int, img_h: int,
                    valid_classes: set[int] | None = None) -> tuple[list[Annotation], int]:
    anns: list[Annotation] = []
    skipped = 0
    path = Path(path)
    if not path.exists():
        return anns, 0
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if not line.strip():
            continue
        ann = parse_line(line, img_w, img_h)
        if ann is None or (valid_classes is not None and ann.cls_id not in valid_classes):
            skipped += 1
            continue
        anns.append(ann)
    return anns, skipped


In [ ]:
%%writefile src/masks.py
from __future__ import annotations

import cv2
import numpy as np

from .labels import Annotation


def polygon_mask(ann: Annotation, h: int, w: int) -> np.ndarray:
    m = np.zeros((h, w), dtype=np.uint8)
    pts = np.round(ann.polygon).astype(np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(m, [pts], 1)
    return m.astype(bool)


def compose_semantic_mask(object_masks: list[tuple[int, np.ndarray]], h: int, w: int) -> np.ndarray:
    sem = np.zeros((h, w), dtype=np.uint8)
    for cls_idx, m in sorted(object_masks, key=lambda t: -int(t[1].sum())):
        sem[m] = cls_idx
    return sem


class SamRefiner:

    def __init__(self, weights: str = "sam2.1_b.pt", device: str | None = None,
                 dilate_px: int = 6, min_ratio: float = 0.03):
        from ultralytics import SAM
        self.model = SAM(weights)
        self.device = device
        self.dilate_px = dilate_px
        self.min_ratio = min_ratio

    def refine(self, image_bgr: np.ndarray, anns: list[Annotation]) -> list[tuple[np.ndarray, str]]:
        h, w = image_bgr.shape[:2]
        poly_masks = [polygon_mask(a, h, w) for a in anns]
        if not anns:
            return []
        boxes = [list(a.bbox) for a in anns]
        kwargs = {"bboxes": boxes, "verbose": False}
        if self.device:
            kwargs["device"] = self.device
        try:
            res = self.model(image_bgr, **kwargs)[0]
            sam_masks = res.masks.data.cpu().numpy() if res.masks is not None else None
        except Exception as exc:
            print(f"  SAM error: {exc}; используется рамка")
            sam_masks = None

        kernel = np.ones((2 * self.dilate_px + 1, 2 * self.dilate_px + 1), np.uint8)
        out = []
        for i, pm in enumerate(poly_masks):
            if sam_masks is None or i >= len(sam_masks):
                out.append((pm, "box"))
                continue
            sm = sam_masks[i]
            if sm.shape != (h, w):
                sm = cv2.resize(sm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            region = cv2.dilate(pm.astype(np.uint8), kernel).astype(bool)
            refined = sm.astype(bool) & region
            ratio = refined.sum() / max(pm.sum(), 1)
            if ratio < self.min_ratio:
                out.append((pm, "box"))
            else:
                out.append((refined, "sam"))
        return out


_FONT_CACHE: dict = {}


def put_text(image_rgb: np.ndarray, text: str, xy: tuple[int, int], color=(255, 255, 255),
             size: int = 18) -> np.ndarray:
    from PIL import Image, ImageDraw, ImageFont
    if size not in _FONT_CACHE:
        try:
            from matplotlib import font_manager
            _FONT_CACHE[size] = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), size)
        except Exception:
            _FONT_CACHE[size] = ImageFont.load_default()
    im = Image.fromarray(image_rgb)
    ImageDraw.Draw(im).text(xy, text, fill=tuple(color), font=_FONT_CACHE[size],
                            stroke_width=2, stroke_fill=(0, 0, 0))
    return np.asarray(im).copy()


def colorize(mask: np.ndarray, colors: dict[int, tuple[int, int, int]]) -> np.ndarray:
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for k, c in colors.items():
        rgb[mask == k] = c
    return rgb


def overlay(image_rgb: np.ndarray, mask: np.ndarray, colors: dict[int, tuple[int, int, int]],
            alpha: float = 0.5) -> np.ndarray:
    out = image_rgb.copy()
    col = colorize(mask, colors)
    fg = mask > 0
    out[fg] = (alpha * col[fg] + (1 - alpha) * out[fg]).astype(np.uint8)
    return out


In [ ]:
%%writefile src/data.py
from __future__ import annotations

import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from .config import IMAGE_EXTS, IMAGENET_MEAN, IMAGENET_STD

SPLIT_ALIASES = {"train": "train", "training": "train", "val": "val", "valid": "val",
                 "validation": "val", "test": "test", "testing": "test"}


def find_pairs(root: str | Path) -> list[tuple[Path, Path | None]]:
    root = Path(root)
    images, labels = [], {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            p = Path(dirpath) / f
            ext = p.suffix.lower()
            if ext in IMAGE_EXTS:
                images.append(p)
            elif ext == ".txt" and p.stem.lower() not in {"classes", "readme", "notes"}:
                labels.setdefault(p.stem, []).append(p)

    pairs = []
    for img in sorted(images):
        cands = labels.get(img.stem, [])
        lab = None
        if len(cands) == 1:
            lab = cands[0]
        elif len(cands) > 1:
            mirror = Path(str(img.with_suffix(".txt")).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}"))
            lab = mirror if mirror in cands else cands[0]
        pairs.append((img, lab))
    return pairs


def detect_split(path: Path) -> str | None:
    for part in path.parts:
        s = SPLIT_ALIASES.get(part.lower())
        if s:
            return s
    return None


def make_split(n: int, val_frac: float, test_frac: float, seed: int) -> list[str]:
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_test = int(round(n * test_frac))
    n_val = int(round(n * val_frac))
    split = ["train"] * n
    for i in idx[:n_test]:
        split[i] = "test"
    for i in idx[n_test:n_test + n_val]:
        split[i] = "val"
    return split


def get_transforms(img_h: int, img_w: int, train: bool):
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    if train:
        aug = [
            A.Resize(img_h, img_w),
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.85, 1.2), translate_percent=(-0.05, 0.05), rotate=(-7, 7), p=0.6),
            A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
            A.HueSaturationValue(8, 15, 10, p=0.4),
            A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=(3, 5))], p=0.2),
            A.CLAHE(p=0.1),
        ]
    else:
        aug = [A.Resize(img_h, img_w)]
    aug += [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return A.Compose(aug)


class RoadDefectDataset:

    def __init__(self, index_csv: str | Path, split: str, transform=None, root: str | Path | None = None):
        df = pd.read_csv(index_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.transform = transform
        self.root = Path(root) if root else Path(index_csv).parent

    def __len__(self):
        return len(self.df)

    def _resolve(self, p: str) -> Path:
        p = Path(p)
        return p if p.is_absolute() else self.root / p

    def load_raw(self, i: int) -> tuple[np.ndarray, np.ndarray]:
        row = self.df.iloc[i]
        img = cv2.imread(str(self._resolve(row["image_path"])), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self._resolve(row["mask_path"])), cv2.IMREAD_GRAYSCALE)
        return img, mask

    def __getitem__(self, i: int):
        import torch
        img, mask = self.load_raw(i)
        if self.transform is not None:
            out = self.transform(image=img, mask=mask)
            img, mask = out["image"], out["mask"]
        return img, torch.as_tensor(np.asarray(mask), dtype=torch.long)


In [ ]:
%%writefile src/metrics.py
from __future__ import annotations

import numpy as np


class SegMetrics:
    def __init__(self, num_classes: int, class_names: list[str] | None = None,
                 defect_ids: list[int] | None = None):
        self.n = num_classes
        self.names = class_names or [str(i) for i in range(num_classes)]
        self.defect_ids = defect_ids if defect_ids is not None else list(range(1, num_classes))
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.n, self.n), dtype=np.int64)

    def update(self, pred, target):
        if hasattr(pred, "detach"):
            pred = pred.detach().cpu().numpy()
        if hasattr(target, "detach"):
            target = target.detach().cpu().numpy()
        p = np.asarray(pred).reshape(-1).astype(np.int64)
        t = np.asarray(target).reshape(-1).astype(np.int64)
        valid = (t >= 0) & (t < self.n)
        self.cm += np.bincount(self.n * t[valid] + p[valid], minlength=self.n ** 2).reshape(self.n, self.n)

    @staticmethod
    def _safe(num, den):
        return float(num / den) if den > 0 else float("nan")

    def per_class(self) -> dict[str, dict[str, float]]:
        res = {}
        total = self.cm.sum()
        for c in range(self.n):
            tp = self.cm[c, c]
            fp = self.cm[:, c].sum() - tp
            fn = self.cm[c, :].sum() - tp
            res[self.names[c]] = {
                "iou": self._safe(tp, tp + fp + fn),
                "dice": self._safe(2 * tp, 2 * tp + fp + fn),
                "precision": self._safe(tp, tp + fp),
                "recall": self._safe(tp, tp + fn),
                "support_px": int(tp + fn),
                "share_px": self._safe(tp + fn, total),
            }
        return res

    def binary(self) -> dict[str, float]:
        d = self.defect_ids
        tp = self.cm[np.ix_(d, d)].sum()
        fp = self.cm[0, d].sum()
        fn = self.cm[d, 0].sum()
        return {
            "iou": self._safe(tp, tp + fp + fn),
            "dice": self._safe(2 * tp, 2 * tp + fp + fn),
            "precision": self._safe(tp, tp + fp),
            "recall": self._safe(tp, tp + fn),
        }

    def summary(self) -> dict:
        pc = self.per_class()
        defect_names = [self.names[i] for i in self.defect_ids]

        def mean_over(key, names):
            vals = [pc[n][key] for n in names if not np.isnan(pc[n][key])]
            return float(np.mean(vals)) if vals else float("nan")

        return {
            "pixel_accuracy": self._safe(np.trace(self.cm), self.cm.sum()),
            "mIoU": mean_over("iou", defect_names),
            "mDice": mean_over("dice", defect_names),
            "mPrecision": mean_over("precision", defect_names),
            "mRecall": mean_over("recall", defect_names),
            "mIoU_with_bg": mean_over("iou", self.names),
            "binary_defect": self.binary(),
            "per_class": pc,
            "confusion_matrix": self.cm.tolist(),
        }


In [ ]:
%%writefile src/model.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

ARCHS = {
    "unet": "Unet",
    "unetplusplus": "UnetPlusPlus",
    "deeplabv3plus": "DeepLabV3Plus",
    "fpn": "FPN",
    "pspnet": "PSPNet",
}


def build_model(arch: str = "unet", encoder: str = "resnet34", num_classes: int = 4,
                encoder_weights: str | None = "imagenet") -> nn.Module:
    import segmentation_models_pytorch as smp

    key = arch.lower().replace("-", "").replace("_", "")
    if key not in ARCHS:
        raise ValueError(f"Неизвестная архитектура {arch}; доступны: {list(ARCHS)}")
    cls = getattr(smp, ARCHS[key])
    return cls(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes)


class DiceCELoss(nn.Module):

    def __init__(self, num_classes: int, class_weights: torch.Tensor | None = None,
                 dice_weight: float = 1.0, ce_weight: float = 1.0, smooth: float = 1.0):
        super().__init__()
        self.n = num_classes
        self.register_buffer("w", class_weights if class_weights is not None else torch.ones(num_classes))
        self.dice_weight, self.ce_weight, self.smooth = dice_weight, ce_weight, smooth

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, target, weight=self.w.to(logits.dtype))
        probs = logits.float().softmax(1)
        onehot = F.one_hot(target, self.n).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        inter = (probs * onehot).sum(dims)
        card = probs.sum(dims) + onehot.sum(dims)
        dice = (2 * inter + self.smooth) / (card + self.smooth)
        present = onehot.sum(dims) > 0
        present[0] = True
        dice_loss = 1 - dice[present].mean()
        return self.ce_weight * ce + self.dice_weight * dice_loss


def compute_class_weights(pixel_counts: list[int] | torch.Tensor, max_w: float = 10.0) -> torch.Tensor:
    c = torch.as_tensor(pixel_counts, dtype=torch.float64).clamp(min=1)
    freq = c / c.sum()
    w = 1.0 / freq.sqrt()
    w = w / w[0]
    return w.clamp(max=max_w).float()


In [ ]:
%%writefile src/inference.py
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import torch

from .config import IMAGENET_MEAN, IMAGENET_STD
from .model import build_model


def load_model(ckpt_path: str | Path, device: torch.device | None = None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    meta = ckpt["meta"]
    model = build_model(meta["arch"], meta["encoder"], meta["num_classes"], encoder_weights=None)
    model.load_state_dict(ckpt["model"])
    model.to(device).eval()
    return model, meta, device


def preprocess(image_rgb: np.ndarray, img_h: int, img_w: int) -> torch.Tensor:
    x = cv2.resize(image_rgb, (img_w, img_h), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
    x = (x - np.array(IMAGENET_MEAN, np.float32)) / np.array(IMAGENET_STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1))


def remove_small_regions(mask: np.ndarray, min_area: int, class_ids) -> tuple[np.ndarray, dict[int, int]]:
    mask = mask.copy()
    counts = {}
    for c in class_ids:
        n, lab, st, _ = cv2.connectedComponentsWithStats((mask == c).astype(np.uint8))
        small = [k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] < min_area]
        if small:
            mask[np.isin(lab, small)] = 0
        counts[c] = n - 1 - len(small)
    return mask, counts


@torch.no_grad()
def predict_probs(model, batch: torch.Tensor, device, tta: bool = False) -> torch.Tensor:
    batch = batch.to(device)
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        probs = model(batch).float().softmax(1)
        if tta:
            probs = (probs + model(batch.flip(-1)).float().softmax(1).flip(-1)) / 2
    return probs


def predict_mask(model, meta, device, image_rgb: np.ndarray, tta: bool = False) -> tuple[np.ndarray, np.ndarray]:
    h, w = image_rgb.shape[:2]
    x = preprocess(image_rgb, meta["img_h"], meta["img_w"]).unsqueeze(0)
    probs = predict_probs(model, x, device, tta)[0]
    probs = torch.nn.functional.interpolate(probs[None], size=(h, w), mode="bilinear", align_corners=False)[0]
    conf, mask = probs.max(0)
    return mask.cpu().numpy().astype(np.uint8), conf.cpu().numpy()


In [ ]:
%%writefile scripts/prepare_dataset.py
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import CLASS_COLORS, CLASS_NAMES, NUM_CLASSES, SEED, SOURCE_CLASSES
from src.data import detect_split, find_pairs, make_split
from src.labels import read_label_file
from src.masks import SamRefiner, compose_semantic_mask, overlay, polygon_mask


def parse_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-root", required=True, help="папка датасета (ищется рекурсивно)")
    ap.add_argument("--out", default="prepared", help="куда сохранить маски и index.csv")
    ap.add_argument("--mask-mode", choices=["box", "sam"], default="sam",
                    help="box — залитые рамки; sam — уточнение рамок моделью SAM 2")
    ap.add_argument("--sam-weights", default="sam2.1_b.pt")
    ap.add_argument("--device", default=None, help="cuda / cpu для SAM")
    ap.add_argument("--val", type=float, default=0.15)
    ap.add_argument("--test", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--use-existing-split", action="store_true",
                    help="использовать папки train/val/test датасета, если они есть")
    ap.add_argument("--keep-unlabeled", action="store_true",
                    help="изображения без txt включать как «без дефектов»")
    ap.add_argument("--vis", type=int, default=12, help="сколько примеров разметки сохранить")
    ap.add_argument("--limit", type=int, default=0, help="обработать только N изображений (отладка)")
    return ap.parse_args()


def main():
    args = parse_args()
    out = Path(args.out)
    (out / "masks").mkdir(parents=True, exist_ok=True)
    (out / "examples").mkdir(exist_ok=True)

    pairs = find_pairs(args.data_root)
    print(f"Найдено изображений: {len(pairs)}; с разметкой: {sum(l is not None for _, l in pairs)}")
    if not args.keep_unlabeled:
        pairs = [(i, l) for i, l in pairs if l is not None]
    if args.limit:
        pairs = pairs[: args.limit]
    if not pairs:
        sys.exit("Нет пар изображение+разметка. Проверьте --data-root.")

    refiner = SamRefiner(args.sam_weights, device=args.device) if args.mask_mode == "sam" else None
    valid = set(SOURCE_CLASSES)

    rows, used_stems = [], Counter()
    obj_counts = Counter()
    src_counts = Counter()
    bad_lines = 0
    for k, (img_path, lab_path) in enumerate(tqdm(pairs, desc="маски")):
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"  не читается: {img_path}")
            continue
        h, w = img.shape[:2]
        anns, skipped = read_label_file(lab_path, w, h, valid) if lab_path else ([], 0)
        bad_lines += skipped

        if refiner is not None and anns:
            masks_src = refiner.refine(img, anns)
        else:
            masks_src = [(polygon_mask(a, h, w), "box") for a in anns]
        obj_masks = []
        for a, (m, src) in zip(anns, masks_src):
            obj_masks.append((a.cls_id + 1, m))
            obj_counts[a.cls_id + 1] += 1
            src_counts[(a.cls_id + 1, src)] += 1
        sem = compose_semantic_mask(obj_masks, h, w)

        stem = img_path.stem
        used_stems[stem] += 1
        name = stem if used_stems[stem] == 1 else f"{stem}__{used_stems[stem]}"
        mask_rel = f"masks/{name}.png"
        cv2.imwrite(str(out / mask_rel), sem)

        px = np.bincount(sem.reshape(-1), minlength=NUM_CLASSES)
        row = {"image_path": str(img_path.resolve()), "mask_path": mask_rel,
               "width": w, "height": h, "orig_split": detect_split(img_path) or ""}
        for c in range(1, NUM_CLASSES):
            row[f"n_{CLASS_NAMES[c]}"] = sum(1 for cid, _ in obj_masks if cid == c)
        for c in range(NUM_CLASSES):
            row[f"px_{CLASS_NAMES[c]}"] = int(px[c])
        rows.append(row)

        if k < args.vis:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            vis = overlay(rgb, sem, CLASS_COLORS, alpha=0.55)
            for a in anns:
                cv2.polylines(vis, [np.round(a.polygon).astype(np.int32)], True, (255, 255, 255), 1)
            both = np.concatenate([rgb, vis], axis=1)
            cv2.imwrite(str(out / "examples" / f"{name}.jpg"), cv2.cvtColor(both, cv2.COLOR_RGB2BGR))

    df = pd.DataFrame(rows)
    has_orig = args.use_existing_split and (df["orig_split"] != "").all() and df["orig_split"].nunique() >= 2
    if has_orig:
        df["split"] = df["orig_split"]
        if "val" not in set(df["split"]):
            tr = df.index[df["split"] == "train"].tolist()
            sub = make_split(len(tr), args.val / (1 - args.test), 0.0, args.seed)
            for i, s in zip(tr, sub):
                df.at[i, "split"] = s
    else:
        df["split"] = make_split(len(df), args.val, args.test, args.seed)
    df.to_csv(out / "index.csv", index=False)

    stats = {"mask_mode": args.mask_mode, "images": len(df), "skipped_label_lines": bad_lines,
             "split_sizes": df["split"].value_counts().to_dict(), "objects": {}, "mask_source": {},
             "pixels_train": {}}
    for c in range(1, NUM_CLASSES):
        stats["objects"][CLASS_NAMES[c]] = int(obj_counts[c])
        stats["mask_source"][CLASS_NAMES[c]] = {s: int(src_counts[(c, s)]) for s in ("sam", "box")}
    per_split = defaultdict(dict)
    for s, g in df.groupby("split"):
        for c in range(1, NUM_CLASSES):
            per_split[s][CLASS_NAMES[c]] = int(g[f"n_{CLASS_NAMES[c]}"].sum())
    stats["objects_per_split"] = per_split
    tr = df[df["split"] == "train"]
    stats["pixels_train"] = {CLASS_NAMES[c]: int(tr[f"px_{CLASS_NAMES[c]}"].sum()) for c in range(NUM_CLASSES)}
    (out / "dataset_stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(stats, ensure_ascii=False, indent=2))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from src.config import CLASS_NAMES_RU
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        names = CLASS_NAMES_RU[1:]
        ax[0].bar(names, [obj_counts[c] for c in range(1, NUM_CLASSES)],
                  color=[np.array(CLASS_COLORS[c]) / 255 for c in range(1, NUM_CLASSES)])
        ax[0].set_title("Число объектов по классам")
        px = [stats["pixels_train"][CLASS_NAMES[c]] for c in range(NUM_CLASSES)]
        ax[1].bar(CLASS_NAMES_RU, np.array(px) / max(sum(px), 1) * 100,
                  color=[np.array(CLASS_COLORS[c]) / 255 if c else (0.6, 0.6, 0.6) for c in range(NUM_CLASSES)])
        ax[1].set_yscale("log")
        ax[1].set_title("Доля пикселей классов, % (train, лог. шкала)")
        fig.tight_layout()
        fig.savefig(out / "class_distribution.png", dpi=150)
    except Exception as exc:
        print("График не построен:", exc)
    print(f"Готово: {out / 'index.csv'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train.py
from __future__ import annotations

import argparse
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import (CLASS_NAMES, DEFAULT_IMG_H, DEFAULT_IMG_W, DEFECT_CLASS_IDS,
                        NUM_CLASSES, SEED)
from src.data import RoadDefectDataset, get_transforms
from src.metrics import SegMetrics
from src.model import DiceCELoss, build_model, compute_class_weights


def parse_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default="prepared", help="папка с index.csv (результат prepare_dataset.py)")
    ap.add_argument("--out", default="runs/unet_resnet34")
    ap.add_argument("--arch", default="unet", help="unet | unetplusplus | deeplabv3plus | fpn | pspnet")
    ap.add_argument("--encoder", default="resnet34")
    ap.add_argument("--no-pretrained", action="store_true", help="не загружать веса ImageNet")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--img-h", type=int, default=DEFAULT_IMG_H)
    ap.add_argument("--img-w", type=int, default=DEFAULT_IMG_W)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--patience", type=int, default=10, help="ранняя остановка по val mIoU")
    ap.add_argument("--class-weights", choices=["auto", "none"], default="auto")
    ap.add_argument("--max-steps", type=int, default=0, help="ограничить число шагов в эпохе (отладка)")
    ap.add_argument("--seed", type=int, default=SEED)
    return ap.parse_args()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate_loader(model, loader, loss_fn, device, max_steps=0):
    model.eval()
    met = SegMetrics(NUM_CLASSES, CLASS_NAMES, DEFECT_CLASS_IDS)
    total, n = 0.0, 0
    for step, (x, y) in enumerate(loader):
        if max_steps and step >= max_steps:
            break
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(x)
        total += loss_fn(logits.float(), y).item() * x.size(0)
        n += x.size(0)
        met.update(logits.argmax(1), y)
    return total / max(n, 1), met.summary()


def plot_history(hist: pd.DataFrame, path: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(hist.epoch, hist.train_loss, label="train")
    ax[0].plot(hist.epoch, hist.val_loss, label="val")
    ax[0].set_title("Функция потерь"); ax[0].legend(); ax[0].set_xlabel("эпоха")
    ax[1].plot(hist.epoch, hist.val_mIoU, label="mIoU")
    ax[1].plot(hist.epoch, hist.val_mDice, label="mDice")
    ax[1].set_title("Валидация (среднее по дефектам)"); ax[1].legend(); ax[1].set_xlabel("эпоха")
    for c in DEFECT_CLASS_IDS:
        ax[2].plot(hist.epoch, hist[f"val_iou_{CLASS_NAMES[c]}"], label=CLASS_NAMES[c])
    ax[2].set_title("IoU по классам (val)"); ax[2].legend(); ax[2].set_xlabel("эпоха")
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main():
    args = parse_args()
    set_seed(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Устройство:", device)

    index = Path(args.data) / "index.csv"
    train_ds = RoadDefectDataset(index, "train", get_transforms(args.img_h, args.img_w, True))
    val_ds = RoadDefectDataset(index, "val", get_transforms(args.img_h, args.img_w, False))
    print(f"train: {len(train_ds)}  val: {len(val_ds)}")
    train_dl = DataLoader(train_ds, args.batch_size, shuffle=True, num_workers=args.workers,
                          pin_memory=device.type == "cuda", drop_last=len(train_ds) > args.batch_size)
    val_dl = DataLoader(val_ds, args.batch_size, shuffle=False, num_workers=args.workers)

    model = build_model(args.arch, args.encoder, NUM_CLASSES,
                        None if args.no_pretrained else "imagenet").to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Модель {args.arch}/{args.encoder}: {n_params / 1e6:.1f} млн параметров")

    if args.class_weights == "auto":
        df = train_ds.df
        counts = [int(df[f"px_{n}"].sum()) for n in CLASS_NAMES]
        weights = compute_class_weights(counts)
    else:
        weights = torch.ones(NUM_CLASSES)
    print("Веса классов:", dict(zip(CLASS_NAMES, [round(float(w), 2) for w in weights])))
    loss_fn = DiceCELoss(NUM_CLASSES, weights).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    steps_per_epoch = min(len(train_dl), args.max_steps) if args.max_steps else len(train_dl)
    total_steps = max(1, args.epochs * steps_per_epoch)
    warmup = min(200, total_steps // 10)

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        prog = (step - warmup) / max(1, total_steps - warmup)
        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * prog))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    meta = {"arch": args.arch, "encoder": args.encoder, "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES, "img_h": args.img_h, "img_w": args.img_w}
    (out / "config.json").write_text(json.dumps({**vars(args), **meta,
                                                 "class_weights": weights.tolist(),
                                                 "params_M": n_params / 1e6}, indent=2, ensure_ascii=False))

    history, best, bad_epochs = [], -1.0, 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, run_loss, seen = time.time(), 0.0, 0
        pbar = tqdm(train_dl, desc=f"эпоха {epoch}/{args.epochs}", leave=False)
        for step, (x, y) in enumerate(pbar):
            if args.max_steps and step >= args.max_steps:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * x.size(0)
            seen += x.size(0)
            pbar.set_postfix(loss=f"{run_loss / seen:.4f}")

        val_loss, s = evaluate_loader(model, val_dl, loss_fn, device, args.max_steps)
        rec = {"epoch": epoch, "lr": opt.param_groups[0]["lr"], "train_loss": run_loss / max(seen, 1),
               "val_loss": val_loss, "val_mIoU": s["mIoU"], "val_mDice": s["mDice"],
               "val_mPrecision": s["mPrecision"], "val_mRecall": s["mRecall"],
               "val_binary_iou": s["binary_defect"]["iou"], "time_s": time.time() - t0}
        for c in DEFECT_CLASS_IDS:
            rec[f"val_iou_{CLASS_NAMES[c]}"] = s["per_class"][CLASS_NAMES[c]]["iou"]
        history.append(rec)
        pd.DataFrame(history).to_csv(out / "history.csv", index=False)
        print(f"эпоха {epoch:3d} | train {rec['train_loss']:.4f} | val {val_loss:.4f} | "
              f"mIoU {s['mIoU']:.4f} | mDice {s['mDice']:.4f} | "
              + " ".join(f"{CLASS_NAMES[c]}={rec[f'val_iou_{CLASS_NAMES[c]}']:.3f}" for c in DEFECT_CLASS_IDS)
              + f" | {rec['time_s']:.0f} c")

        ckpt = {"model": model.state_dict(), "meta": meta, "epoch": epoch, "val": s}
        torch.save(ckpt, out / "last.pt")
        score = s["mIoU"] if not np.isnan(s["mIoU"]) else 0.0
        if score > best:
            best, bad_epochs = score, 0
            torch.save(ckpt, out / "best.pt")
            print(f"   ↑ лучшая модель сохранена (val mIoU = {best:.4f})")
        else:
            bad_epochs += 1
            if bad_epochs >= args.patience:
                print(f"Ранняя остановка: {args.patience} эпох без улучшения")
                break

    plot_history(pd.DataFrame(history), out / "training_curves.png")
    print(f"Лучший val mIoU: {best:.4f}. Файлы: {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/evaluate.py
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import CLASS_COLORS, CLASS_NAMES, CLASS_NAMES_RU, DEFECT_CLASS_IDS, NUM_CLASSES
from src.data import RoadDefectDataset
from src.inference import load_model, predict_probs, preprocess
from src.masks import overlay, put_text
from src.metrics import SegMetrics


def parse_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data", default="prepared")
    ap.add_argument("--split", default="test")
    ap.add_argument("--out", default=None, help="по умолчанию <папка чекпойнта>/eval_<split>")
    ap.add_argument("--tta", action="store_true", help="усреднение с отражённым изображением")
    ap.add_argument("--n-vis", type=int, default=12)
    ap.add_argument("--full-res", action="store_true",
                    help="считать метрики в исходном разрешении (иначе — во входном разрешении сети)")
    return ap.parse_args()


def save_confusion(cm: np.ndarray, path: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES_RU)
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES_RU)
    ax.set_xlabel("Прогноз"); ax.set_ylabel("Истина")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title("Нормированная матрица ошибок (по пикселям)")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def save_bars(table: pd.DataFrame, path: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    t = table[table["class"] != "background"]
    keys = ["iou", "dice", "precision", "recall"]
    x = np.arange(len(t))
    fig, ax = plt.subplots(figsize=(8, 4))
    for k, key in enumerate(keys):
        ax.bar(x + (k - 1.5) * 0.2, t[key], 0.2, label=key)
    ax.set_xticks(x, t["class_ru"]); ax.set_ylim(0, 1); ax.legend(ncols=4); ax.grid(axis="y", alpha=0.3)
    ax.set_title("Метрики сегментации по классам (тест)")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def main():
    args = parse_args()
    model, meta, device = load_model(args.ckpt)
    out = Path(args.out) if args.out else Path(args.ckpt).parent / f"eval_{args.split}"
    (out / "vis").mkdir(parents=True, exist_ok=True)
    ds = RoadDefectDataset(Path(args.data) / "index.csv", args.split, transform=None)
    print(f"{args.split}: {len(ds)} изображений, модель {meta['arch']}/{meta['encoder']}, устройство {device}")

    met = SegMetrics(NUM_CLASSES, CLASS_NAMES, DEFECT_CLASS_IDS)
    per_image, times = [], []
    for i in tqdm(range(len(ds)), desc="тест"):
        img, gt = ds.load_raw(i)
        x = preprocess(img, meta["img_h"], meta["img_w"]).unsqueeze(0)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        probs = predict_probs(model, x, device, args.tta)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

        if args.full_res:
            probs = F.interpolate(probs, size=gt.shape, mode="bilinear", align_corners=False)
            target = gt
        else:
            target = cv2.resize(gt, (meta["img_w"], meta["img_h"]), interpolation=cv2.INTER_NEAREST)
        pred = probs.argmax(1)[0].cpu().numpy().astype(np.uint8)
        met.update(pred, target)

        im_met = SegMetrics(NUM_CLASSES, CLASS_NAMES, DEFECT_CLASS_IDS)
        im_met.update(pred, target)
        b = im_met.binary()
        per_image.append({"image": Path(ds.df.iloc[i]["image_path"]).name, "defect_iou": b["iou"],
                          "gt_defect_px": int((target > 0).sum()), "pred_defect_px": int((pred > 0).sum())})

        if i < args.n_vis:
            small = cv2.resize(img, (pred.shape[1], pred.shape[0]))
            panel = np.concatenate([small, overlay(small, target, CLASS_COLORS, 0.6),
                                    overlay(small, pred, CLASS_COLORS, 0.6)], axis=1)
            for k, title in enumerate(["Изображение", "Разметка", "Прогноз"]):
                panel = put_text(panel, title, (k * small.shape[1] + 8, 6))
            cv2.imwrite(str(out / "vis" / f"{i:03d}.jpg"), cv2.cvtColor(panel, cv2.COLOR_RGB2BGR))

    s = met.summary()
    warm = times[3:] if len(times) > 5 else times
    s["inference_ms_per_image"] = float(np.mean(warm) * 1000)
    s["fps"] = float(1.0 / np.mean(warm))
    s["tta"] = args.tta
    s["split"] = args.split
    s["n_images"] = len(ds)
    s["model"] = f"{meta['arch']}/{meta['encoder']}"
    (out / "metrics.json").write_text(json.dumps(s, indent=2, ensure_ascii=False), encoding="utf-8")

    rows = [{"class": n, "class_ru": CLASS_NAMES_RU[k], **{m: s["per_class"][n][m] for m in
             ("iou", "dice", "precision", "recall", "share_px")}} for k, n in enumerate(CLASS_NAMES)]
    rows.append({"class": "mean_defects", "class_ru": "среднее (дефекты)", "iou": s["mIoU"], "dice": s["mDice"],
                 "precision": s["mPrecision"], "recall": s["mRecall"], "share_px": np.nan})
    rows.append({"class": "defect_vs_bg", "class_ru": "дефект/фон", **s["binary_defect"], "share_px": np.nan})
    table = pd.DataFrame(rows)
    table.to_csv(out / "metrics_table.csv", index=False, float_format="%.4f")
    pd.DataFrame(per_image).to_csv(out / "per_image.csv", index=False, float_format="%.4f")
    save_confusion(np.array(s["confusion_matrix"]), out / "confusion_matrix.png")
    save_bars(table, out / "metrics_by_class.png")

    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"Pixel accuracy: {s['pixel_accuracy']:.4f} | время: {s['inference_ms_per_image']:.1f} мс/кадр "
          f"({s['fps']:.1f} FPS)")
    print(f"Результаты: {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/predict.py
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import CLASS_COLORS, CLASS_NAMES, CLASS_NAMES_RU, DEFECT_CLASS_IDS, IMAGE_EXTS
from src.inference import load_model, predict_mask, remove_small_regions
from src.knowledge import InferenceEngine, KnowledgeBase, extract_facts
from src.masks import overlay, put_text


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--input", required=True, help="изображение или папка")
    ap.add_argument("--out", default="predictions")
    ap.add_argument("--tta", action="store_true")
    ap.add_argument("--kb", default=None, help="файл базы знаний")
    ap.add_argument("--min-area", type=int, default=30, help="мин. площадь области в пикселях")
    args = ap.parse_args()

    model, meta, device = load_model(args.ckpt)
    kb = KnowledgeBase(args.kb) if args.kb else KnowledgeBase()
    engine = InferenceEngine(kb)
    inp = Path(args.input)
    files = [inp] if inp.is_file() else sorted(p for p in inp.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    out = Path(args.out)
    (out / "masks").mkdir(parents=True, exist_ok=True)
    (out / "overlay").mkdir(exist_ok=True)

    rows = []
    for p in files:
        bgr = cv2.imread(str(p))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mask, conf = predict_mask(model, meta, device, rgb, args.tta)

        mask, regions = remove_small_regions(mask, args.min_area, DEFECT_CLASS_IDS)
        row = {"image": p.name}
        for c in DEFECT_CLASS_IDS:
            row[f"{CLASS_NAMES[c]}_regions"] = regions[c]
            row[f"{CLASS_NAMES[c]}_area_%"] = round(100 * float((mask == c).mean()), 3)
        res = engine.run(extract_facts(mask, conf, kb.params.get("near_zone_from", 0.6)))
        row["state"] = res.level_name
        row["priority"] = res.priority
        row["rules"] = ",".join(f["id"] for f in res.fired)
        row["recommendations"] = " | ".join(res.recommendations)
        row["mean_conf_defect"] = round(float(conf[mask > 0].mean()), 3) if (mask > 0).any() else None
        rows.append(row)

        cv2.imwrite(str(out / "masks" / f"{p.stem}.png"), mask)
        vis = overlay(rgb, mask, CLASS_COLORS, 0.55)
        vis = put_text(vis, f"Состояние: {res.level_name}", (8, vis.shape[0] - 26), (255, 255, 255), 17)
        y = 6
        for c in DEFECT_CLASS_IDS:
            if row[f"{CLASS_NAMES[c]}_regions"]:
                txt = (f"{CLASS_NAMES_RU[c]}: {row[f'{CLASS_NAMES[c]}_regions']} обл., "
                       f"{row[f'{CLASS_NAMES[c]}_area_%']}% кадра")
                vis = put_text(vis, txt, (8, y), CLASS_COLORS[c], 16)
                y += 22
        cv2.imwrite(str(out / "overlay" / f"{p.stem}.jpg"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))

    df = pd.DataFrame(rows)
    df.to_csv(out / "summary.csv", index=False)
    print(df.to_string(index=False))
    print("Легенда:", ", ".join(f"{CLASS_NAMES_RU[c]} — {CLASS_NAMES[c]}" for c in DEFECT_CLASS_IDS))
    print(f"Сохранено в {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/compare_models.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+", help="папки запусков (внутри eval_test/metrics.json)")
    ap.add_argument("--split", default="test")
    ap.add_argument("--out", default="runs/comparison.csv")
    args = ap.parse_args()

    rows = []
    for r in args.runs:
        r = Path(r)
        mfile = r / f"eval_{args.split}" / "metrics.json"
        if not mfile.exists():
            print("нет", mfile)
            continue
        m = json.loads(mfile.read_text(encoding="utf-8"))
        cfg = json.loads((r / "config.json").read_text(encoding="utf-8")) if (r / "config.json").exists() else {}
        row = {"модель": m["model"], "параметров, млн": round(cfg.get("params_M", float("nan")), 1),
               "mIoU": m["mIoU"], "mDice": m["mDice"], "mPrecision": m["mPrecision"], "mRecall": m["mRecall"],
               "IoU выбоина": m["per_class"]["pothole"]["iou"], "IoU трещина": m["per_class"]["crack"]["iou"],
               "IoU люк": m["per_class"]["manhole"]["iou"], "IoU дефект/фон": m["binary_defect"]["iou"],
               "мс/кадр": m["inference_ms_per_image"]}
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("mIoU", ascending=False)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.out, index=False, float_format="%.4f")
    print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/evaluate_rules.py
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import DEFECT_CLASS_IDS
from src.data import RoadDefectDataset
from src.inference import load_model, predict_mask, remove_small_regions
from src.knowledge import InferenceEngine, KnowledgeBase, extract_facts


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data", default="prepared")
    ap.add_argument("--split", default="test")
    ap.add_argument("--kb", default=None)
    ap.add_argument("--min-area", type=int, default=30)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    kb = KnowledgeBase(args.kb) if args.kb else KnowledgeBase()
    engine = InferenceEngine(kb)
    nz = kb.params.get("near_zone_from", 0.6)
    model, meta, device = load_model(args.ckpt)
    ds = RoadDefectDataset(Path(args.data) / "index.csv", args.split)
    out = Path(args.out) if args.out else Path(args.ckpt).parent / f"rules_{args.split}"
    out.mkdir(parents=True, exist_ok=True)

    rows, fired_gt, fired_pred = [], Counter(), Counter()
    for i in tqdm(range(len(ds)), desc="правила"):
        img, gt = ds.load_raw(i)
        pred, conf = predict_mask(model, meta, device, img)
        pred, _ = remove_small_regions(pred, args.min_area, DEFECT_CLASS_IDS)
        gt_c, _ = remove_small_regions(gt, args.min_area, DEFECT_CLASS_IDS)
        r_gt = engine.run(extract_facts(gt_c, None, nz))
        r_pr = engine.run(extract_facts(pred, conf, nz))
        fired_gt.update(f["id"] for f in r_gt.fired)
        fired_pred.update(f["id"] for f in r_pr.fired)
        rows.append({"image": Path(ds.df.iloc[i]["image_path"]).name,
                     "level_gt": r_gt.level, "level_pred": r_pr.level,
                     "state_gt": r_gt.level_name, "state_pred": r_pr.level_name,
                     "warning_gt": bool(r_gt.facts.get("driver_warning")),
                     "warning_pred": bool(r_pr.facts.get("driver_warning"))})

    df = pd.DataFrame(rows)
    df.to_csv(out / "per_image.csv", index=False)
    levels = sorted(kb.levels)
    names = [kb.levels[l]["name"] for l in levels]
    cm = pd.crosstab(pd.Categorical(df.level_gt, levels), pd.Categorical(df.level_pred, levels), dropna=False)
    cm.index, cm.columns = [f"эталон: {n}" for n in names], [f"прогноз: {n}" for n in names]
    cm.to_csv(out / "confusion_levels.csv")

    diff = (df.level_gt - df.level_pred).abs()
    urgent_gt, urgent_pred = df.level_gt >= 3, df.level_pred >= 3
    tp = int((urgent_gt & urgent_pred).sum())
    summary = {
        "n_images": len(df),
        "exact_agreement": float((diff == 0).mean()),
        "within_one_level": float((diff <= 1).mean()),
        "repair_needed_precision": tp / max(int(urgent_pred.sum()), 1),
        "repair_needed_recall": tp / max(int(urgent_gt.sum()), 1),
        "driver_warning_agreement": float((df.warning_gt == df.warning_pred).mean()),
        "distribution_gt": df.state_gt.value_counts().to_dict(),
        "distribution_pred": df.state_pred.value_counts().to_dict(),
        "rules_fired_gt": dict(sorted(fired_gt.items())),
        "rules_fired_pred": dict(sorted(fired_pred.items())),
    }
    (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(cm.to_string())
    print(json.dumps({k: v for k, v in summary.items() if not k.startswith("rules")}, ensure_ascii=False, indent=2))
    print(f"Сохранено в {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/knowledge.py
from __future__ import annotations

import json
import operator
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np

DEFAULT_KB = Path(__file__).resolve().parents[1] / "configs" / "knowledge_base.json"

OPS = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge,
       "==": operator.eq, "!=": operator.ne}

POTHOLE, CRACK, MANHOLE = 1, 2, 3


def extract_facts(mask: np.ndarray, conf: np.ndarray | None = None, near_zone_from: float = 0.6) -> dict:
    h, w = mask.shape
    total = float(h * w)
    facts: dict = {}
    for cid, name in ((POTHOLE, "pothole"), (CRACK, "crack"), (MANHOLE, "manhole")):
        n, lab, st, cen = cv2.connectedComponentsWithStats((mask == cid).astype(np.uint8))
        areas = st[1:, cv2.CC_STAT_AREA].astype(float) if n > 1 else np.zeros(0)
        facts[f"{name}_count"] = int(n - 1)
        facts[f"{name}_area_pct"] = round(100 * areas.sum() / total, 3)
        facts[f"{name}_max_pct"] = round(100 * areas.max() / total, 3) if len(areas) else 0.0
        if name == "pothole":
            near = [a for a, (_, cy) in zip(areas, cen[1:]) if cy >= near_zone_from * h]
            facts["pothole_near_max_pct"] = round(100 * max(near) / total, 3) if near else 0.0
    facts["defect_area_pct"] = round(facts["pothole_area_pct"] + facts["crack_area_pct"], 3)
    defect = (mask == POTHOLE) | (mask == CRACK)
    facts["mean_conf"] = round(float(conf[defect].mean()), 3) if conf is not None and defect.any() else 1.0
    return facts


def _fmt(value, raw=None) -> str:
    text = {True: "да", False: "нет", None: "нет данных"}.get(value, value) if isinstance(value, (bool, type(None))) \
        else f"{value:g}" if isinstance(value, float) else str(value)
    if isinstance(raw, str) and raw.startswith("$"):
        text += f" [{raw[1:]}]"
    return text


@dataclass
class InferenceResult:
    level: int
    level_name: str
    priority: str
    messages: list[str] = field(default_factory=list)
    recommendations: list[str] = field(default_factory=list)
    fired: list[dict] = field(default_factory=list)
    facts: dict = field(default_factory=dict)

    def explanation(self) -> str:
        lines = []
        for f in self.fired:
            conds = " И ".join(f"{fact} {op} {_fmt(val, raw)} (факт: {_fmt(cur)})"
                               for fact, op, raw, cur, val in f["conditions"])
            lines.append(f"{f['id']} «{f['name']}»: ЕСЛИ {conds}")
        return "\n".join(lines)


class KnowledgeBase:
    def __init__(self, path: str | Path = DEFAULT_KB):
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        self.params: dict = data["params"]
        self.param_descriptions: dict = data.get("param_descriptions", {})
        self.rules: list[dict] = data["rules"]
        self.levels: dict = {int(k): v for k, v in data["severity_levels"].items()}
        self._validate()

    def _validate(self):
        ids = [r["id"] for r in self.rules]
        if len(ids) != len(set(ids)):
            raise ValueError("Повторяющиеся id правил в базе знаний")
        for r in self.rules:
            for fact, op, val in r["if"]:
                if op not in OPS:
                    raise ValueError(f"{r['id']}: неизвестная операция {op}")
                if isinstance(val, str) and val.startswith("$") and val[1:] not in self.params:
                    raise ValueError(f"{r['id']}: нет параметра {val}")

    def resolve(self, value):
        if isinstance(value, str) and value.startswith("$"):
            return self.params[value[1:]]
        return value


class InferenceEngine:

    def __init__(self, kb: KnowledgeBase):
        self.kb = kb

    def _check(self, rule: dict, facts: dict):
        used = []
        for fact, op, raw in rule["if"]:
            value = self.kb.resolve(raw)
            current = facts.get(fact)
            if current is None:
                if op == "!=":
                    used.append((fact, op, raw, None, value))
                    continue
                return None
            if not OPS[op](current, value):
                return None
            used.append((fact, op, raw, current, value))
        return used

    def run(self, facts: dict, max_iter: int = 20) -> InferenceResult:
        facts = dict(facts)
        fired_ids: set[str] = set()
        fired, levels, messages, recs = [], [], [], []
        for _ in range(max_iter):
            new_fire = False
            for rule in self.kb.rules:
                if rule["id"] in fired_ids:
                    continue
                used = self._check(rule, facts)
                if used is None:
                    continue
                fired_ids.add(rule["id"])
                new_fire = True
                then = rule["then"]
                facts.update(then.get("facts", {}))
                if "severity" in then:
                    levels.append(then["severity"])
                if then.get("message"):
                    messages.append(then["message"])
                if then.get("recommendation") and then["recommendation"] not in recs:
                    recs.append(then["recommendation"])
                fired.append({"id": rule["id"], "name": rule["name"], "conditions": used,
                              "severity": then.get("severity")})
            if not new_fire:
                break
        level = max(levels) if levels else 0
        info = self.kb.levels[level]
        return InferenceResult(level, info["name"], info["priority"], messages, recs, fired, facts)


def assess(mask: np.ndarray, conf: np.ndarray | None = None, kb: KnowledgeBase | None = None) -> InferenceResult:
    kb = kb or KnowledgeBase()
    facts = extract_facts(mask, conf, kb.params.get("near_zone_from", 0.6))
    return InferenceEngine(kb).run(facts)


In [ ]:
%%writefile src/monitoring.py
from __future__ import annotations

import csv
import html
import json
import math
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np

from .config import CLASS_COLORS, CLASS_NAMES, CLASS_NAMES_RU, DEFECT_CLASS_IDS

LEVEL_COLORS = {0: "#22A447", 1: "#7CB518", 2: "#E0B000", 3: "#F07A10", 4: "#E02424"}


def _to_deg(value) -> float:
    d, m, s = (float(x) for x in value)
    return d + m / 60 + s / 3600


def read_gps(path: str | Path) -> tuple[float, float] | None:
    try:
        from PIL import Image
        with Image.open(path) as im:
            gps = im.getexif().get_ifd(0x8825)
        if not gps or 2 not in gps or 4 not in gps:
            return None
        lat, lon = _to_deg(gps[2]), _to_deg(gps[4])
        if gps.get(1, "N") in ("S", b"S"):
            lat = -lat
        if gps.get(3, "E") in ("W", b"W"):
            lon = -lon
        if not (-90 <= lat <= 90 and -180 <= lon <= 180) or (lat == 0 and lon == 0):
            return None
        return round(lat, 6), round(lon, 6)
    except Exception:
        return None


def parse_coords(text: str | None) -> tuple[float, float] | None:
    if not text or not text.strip():
        return None
    try:
        parts = [float(x) for x in text.replace(";", ",").split(",")]
        if len(parts) == 2 and -90 <= parts[0] <= 90 and -180 <= parts[1] <= 180:
            return round(parts[0], 6), round(parts[1], 6)
    except ValueError:
        pass
    return None


@dataclass
class FrameResult:
    name: str
    overlay: np.ndarray
    mask: np.ndarray
    level: int
    state: str
    priority: str
    areas: dict
    regions: dict
    messages: list
    recommendations: list
    rules: list
    gps: tuple[float, float] | None = None
    time_s: float | None = None

    @property
    def defect_area(self) -> float:
        return self.areas.get("pothole", 0.0) + self.areas.get("crack", 0.0)

    def record(self, source: str) -> dict:
        r = {"время": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "источник": source, "кадр": self.name,
             "состояние": self.state, "уровень": self.level, "приоритет": self.priority}
        for c in DEFECT_CLASS_IDS:
            r[f"{CLASS_NAMES_RU[c]}, областей"] = self.regions[CLASS_NAMES[c]]
            r[f"{CLASS_NAMES_RU[c]}, % кадра"] = self.areas[CLASS_NAMES[c]]
        r["широта"] = self.gps[0] if self.gps else ""
        r["долгота"] = self.gps[1] if self.gps else ""
        r["правила"] = ", ".join(self.rules)
        r["рекомендации"] = " | ".join(self.recommendations)
        return r


class Journal:

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def add(self, records: list[dict]):
        if not records:
            return
        new = not self.path.exists() or self.path.stat().st_size == 0
        fields = list(records[0].keys())
        if not new:
            with self.path.open(encoding="utf-8-sig", newline="") as f:
                fields = next(csv.reader(f), fields)
        with self.path.open("a", encoding="utf-8-sig", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
            if new:
                w.writeheader()
            w.writerows(records)

    def load(self):
        import pandas as pd
        if not self.path.exists() or self.path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(self.path, encoding="utf-8-sig")

    def clear(self):
        if self.path.exists():
            self.path.unlink()


def compare_before_after(before: FrameResult, after: FrameResult, max_ratio: float = 0.3) -> dict:
    a0, a1 = before.defect_area, after.defect_area
    out = {"area_before": round(a0, 2), "area_after": round(a1, 2), "level_before": before.level,
           "level_after": after.level, "ratio": None}
    if a0 <= 0:
        out.update(verdict="сравнение не требуется", detail="На снимке «до» дефекты не обнаружены.")
        return out
    ratio = a1 / a0
    out["ratio"] = round(ratio, 3)
    potholes_after = after.regions.get("pothole", 0)
    if after.level <= 1 or (ratio <= max_ratio and potholes_after == 0):
        out.update(verdict="ремонт выполнен",
                   detail=f"Площадь дефектов уменьшилась с {a0:.2f} % до {a1:.2f} % кадра, выбоин не осталось.")
    elif ratio < 1:
        out.update(verdict="ремонт выполнен частично",
                   detail=f"Площадь дефектов уменьшилась с {a0:.2f} % до {a1:.2f} % кадра "
                          f"({ratio * 100:.0f} % от исходной), выбоин осталось: {potholes_after}.")
    else:
        out.update(verdict="ремонт не выполнен",
                   detail=f"Площадь дефектов не уменьшилась: {a0:.2f} % → {a1:.2f} % кадра.")
    return out


def sample_video(path: str | Path, step_s: float = 1.0, max_frames: int = 300):
    import cv2
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError("Не удалось открыть видео")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(1, int(round(step_s * fps)))
    idx, taken = 0, 0
    try:
        while taken < max_frames and (n == 0 or idx < n):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok:
                break
            yield idx / fps, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            taken += 1
            idx += step
    finally:
        cap.release()


def video_timeline_png(results: list[FrameResult], out_path: str | Path, theme_dark=False):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    t = [r.time_s for r in results]
    fg, bg, grid = ("#e5e5e5", "#0b0b0b", "#2a2a2a") if theme_dark else ("#111111", "#FFFFFF", "#E4E4E1")
    fig, ax = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True, facecolor=bg,
                           gridspec_kw={"height_ratios": [1, 1.2]})
    for a in ax:
        a.set_facecolor(bg); a.tick_params(colors=fg); a.grid(color=grid, lw=0.6)
        for s in a.spines.values():
            s.set_color(grid)
    ax[0].scatter(t, [r.level for r in results], c=[LEVEL_COLORS[r.level] for r in results], s=26, zorder=3)
    ax[0].step(t, [r.level for r in results], where="mid", color="#111111", lw=1.2, alpha=0.5)
    ax[0].set_yticks(range(5)); ax[0].set_ylim(-0.5, 4.5)
    ax[0].set_ylabel("уровень", color=fg)
    for c, col in ((1, CLASS_COLORS[1]), (2, CLASS_COLORS[2]), (3, CLASS_COLORS[3])):
        ax[1].plot(t, [r.areas[CLASS_NAMES[c]] for r in results], color=np.array(col) / 255, lw=1.6,
                   label=CLASS_NAMES_RU[c])
    ax[1].set_ylabel("% кадра", color=fg); ax[1].set_xlabel("время видео, с", color=fg)
    leg = ax[1].legend(facecolor=bg, edgecolor=grid, labelcolor=fg, fontsize=8, ncols=3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130, facecolor=bg)
    plt.close(fig)


def map_html(points: list[dict], height: int = 520) -> str:
    if not points:
        return ("<div style='padding:40px;text-align:center;color:#5B5B57;border:2px dashed #C9C9C4;border-radius:28px'>"
                "Нет кадров с координатами. Координаты берутся из GPS-данных фото (EXIF) "
                "или вводятся вручную на вкладке «Кадр».</div>")
    lat = sum(p["lat"] for p in points) / len(points)
    lon = sum(p["lon"] for p in points) / len(points)
    data = json.dumps([{**p, "color": LEVEL_COLORS.get(int(p["level"]), "#999")} for p in points], ensure_ascii=False)
    fallback = _offline_svg(points, height)
    page = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>html,body,#m{{height:100%;margin:0;background:#F3F3F1;font-family:Manrope,system-ui,sans-serif}}
.leaflet-tile-pane{{filter:grayscale(1) brightness(1.05)}}
.leaflet-popup-content-wrapper,.leaflet-popup-tip{{background:#111111;color:#F3F3F1;border-radius:14px}}
#fb{{display:none;height:100%}}</style>
</head><body><div id="m"></div><div id="fb">{fallback}</div><script>
var pts={data};
if (typeof L === 'undefined') {{ document.getElementById('m').style.display='none';
  document.getElementById('fb').style.display='block'; }}
else {{ var map=L.map('m').setView([{lat},{lon}],14);
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',{{maxZoom:19,attribution:'© OpenStreetMap'}}).addTo(map);
var g=[];pts.forEach(function(p){{var c=L.circleMarker([p.lat,p.lon],{{radius:9,color:p.color,weight:2,fillColor:p.color,fillOpacity:.55}})
.bindPopup('<b>'+p.state+'</b><br>'+p.name+'<br>'+p.time).addTo(map);g.push([p.lat,p.lon]);}});
if(g.length>1)map.fitBounds(g,{{padding:[30,30]}}); }}
</script></body></html>"""
    return (f"<iframe style='width:100%;height:{height}px;border:none;border-radius:28px' "
            f"srcdoc=\"{html.escape(page, quote=True)}\"></iframe>")


def _offline_svg(points: list[dict], height: int) -> str:
    W, H, pad = 1000, height, 60
    lats = [p["lat"] for p in points]; lons = [p["lon"] for p in points]
    la0, la1 = min(lats), max(lats); lo0, lo1 = min(lons), max(lons)
    span = max(la1 - la0, (lo1 - lo0) * math.cos(math.radians(sum(lats) / len(lats))), 1e-4)
    k = min((W - 2 * pad), (H - 2 * pad)) / span
    cx, cy = (lo0 + lo1) / 2, (la0 + la1) / 2
    coslat = math.cos(math.radians(cy))
    def xy(la, lo):
        return W / 2 + (lo - cx) * coslat * k, H / 2 - (la - cy) * k
    grid = "".join(f"<line x1='{x}' y1='0' x2='{x}' y2='{H}' stroke='#E4E4E1'/>" for x in range(0, W, 50)) + \
        "".join(f"<line x1='0' y1='{y}' x2='{W}' y2='{y}' stroke='#E4E4E1'/>" for y in range(0, H, 50))
    marks = ""
    for p in points:
        x, y = xy(p["lat"], p["lon"]); c = LEVEL_COLORS.get(int(p["level"]), "#999")
        marks += (f"<circle cx='{x:.1f}' cy='{y:.1f}' r='16' fill='{c}' opacity='.18'/>"
                  f"<circle cx='{x:.1f}' cy='{y:.1f}' r='7' fill='{c}' stroke='#FFFFFF' stroke-width='2'/>"
                  f"<text x='{x + 14:.1f}' y='{y - 10:.1f}' fill='#111111' font-size='13'>{html.escape(str(p['state']))}</text>"
                  f"<text x='{x + 14:.1f}' y='{y + 6:.1f}' fill='#5B5B57' font-size='11'>{p['lat']:.4f}, {p['lon']:.4f}</text>")
    note = ("<text x='20' y='28' fill='#111111' font-size='12' font-weight='700' letter-spacing='2'>СХЕМА БЕЗ ПОДЛОЖКИ · НЕТ ДОСТУПА К КАРТЕ</text>")
    return (f"<svg viewBox='0 0 {W} {H}' width='100%' height='100%' preserveAspectRatio='xMidYMid meet' "
            f"style='background:#FFFFFF'>{grid}{note}{marks}</svg>")


def pdf_report(results: list[FrameResult], out_path: str | Path, title: str = "Отчет о состоянии покрытия"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages

    A4 = (8.27, 11.69)
    ordered = sorted(results, key=lambda r: -r.level)
    with PdfPages(out_path) as pdf:
        fig = plt.figure(figsize=A4)
        fig.text(0.08, 0.95, title, fontsize=17, weight="bold")
        fig.text(0.08, 0.925, f"Сформирован {datetime.now():%d.%m.%Y %H:%M} · кадров: {len(results)}", fontsize=9,
                 color="#555")
        counts = {}
        for r in results:
            counts[r.state] = counts.get(r.state, 0) + 1
        y = 0.89
        for st, n in sorted(counts.items(), key=lambda kv: -max(r.level for r in results if r.state == kv[0])):
            fig.text(0.08, y, f"{st}: {n}", fontsize=10); y -= 0.02
        rows = [[r.name[:28], r.state, r.priority, f"{r.areas['pothole']:.2f}", f"{r.areas['crack']:.2f}",
                 (f"{r.gps[0]:.5f}, {r.gps[1]:.5f}" if r.gps else "—")] for r in ordered[:40]]
        ax = fig.add_axes([0.06, 0.05, 0.88, y - 0.07]); ax.axis("off")
        if rows:
            t = ax.table(cellText=rows, colLabels=["Кадр", "Состояние", "Приоритет", "Выбоины, %", "Трещины, %",
                                                   "Координаты"], loc="upper center", cellLoc="left")
            t.auto_set_font_size(False); t.set_fontsize(7.5); t.scale(1, 1.25)
            for (i, j), cell in t.get_celld().items():
                cell.set_edgecolor("#bbb")
                if i == 0:
                    cell.set_facecolor("#efefef")
        pdf.savefig(fig); plt.close(fig)
        for r in [r for r in ordered if r.level >= 1][:30]:
            fig = plt.figure(figsize=A4)
            fig.text(0.08, 0.95, r.name, fontsize=13, weight="bold")
            fig.text(0.08, 0.925, f"Состояние: {r.state} · приоритет: {r.priority}"
                     + (f" · координаты: {r.gps[0]:.5f}, {r.gps[1]:.5f}" if r.gps else "")
                     + (f" · время видео: {r.time_s:.1f} с" if r.time_s is not None else ""), fontsize=9)
            ax = fig.add_axes([0.08, 0.5, 0.84, 0.4]); ax.imshow(r.overlay); ax.axis("off")
            y = 0.46
            for c in DEFECT_CLASS_IDS:
                n = CLASS_NAMES[c]
                fig.text(0.08, y, f"{CLASS_NAMES_RU[c]}: областей {r.regions[n]}, {r.areas[n]:.2f} % кадра", fontsize=10)
                y -= 0.022
            y -= 0.01
            for head, items in (("Выводы:", r.messages), ("Рекомендации:", r.recommendations)):
                if items:
                    fig.text(0.08, y, head, fontsize=10, weight="bold"); y -= 0.022
                    for it in items:
                        for line in _wrap(it, 95):
                            fig.text(0.1, y, line, fontsize=9); y -= 0.019
                    y -= 0.01
            fig.text(0.08, y, "Сработавшие правила: " + ", ".join(r.rules), fontsize=8, color="#555")
            pdf.savefig(fig); plt.close(fig)
    return str(out_path)


def _wrap(text: str, width: int) -> list[str]:
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > width:
            lines.append(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur:
        lines.append(cur)
    return lines or [""]


In [ ]:
%%writefile tests/test_monitoring.py
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.monitoring import (FrameResult, Journal, compare_before_after, map_html, parse_coords,
                            pdf_report, read_gps, sample_video)


def fr(level=3, pothole=0.5, crack=2.0, n_pot=1, gps=None, name="k.jpg"):
    return FrameResult(name=name, overlay=np.zeros((36, 64, 3), np.uint8), mask=np.zeros((36, 64), np.uint8),
                       level=level, state=f"уровень {level}", priority="p",
                       areas={"pothole": pothole, "crack": crack, "manhole": 0.0},
                       regions={"pothole": n_pot, "crack": 1 if crack else 0, "manhole": 0},
                       messages=["сообщение"], recommendations=["рекомендация"], rules=["D1", "R4"], gps=gps)


def test_read_gps_from_exif(tmp_path):
    from PIL import Image
    im = Image.new("RGB", (32, 32))
    exif = Image.Exif()
    gps = exif.get_ifd(0x8825)
    gps.update({1: "N", 2: (52.0, 5.0, 51.36), 3: "E", 4: (23.0, 44.0, 2.76)})
    p = tmp_path / "g.jpg"
    im.save(p, exif=exif)
    lat, lon = read_gps(p)
    assert lat == pytest.approx(52.0976, abs=1e-4) and lon == pytest.approx(23.7341, abs=1e-4)


def test_read_gps_absent(tmp_path):
    from PIL import Image
    p = tmp_path / "n.jpg"
    Image.new("RGB", (8, 8)).save(p)
    assert read_gps(p) is None and read_gps(tmp_path / "missing.jpg") is None


@pytest.mark.parametrize("text,expected", [("52.0976, 23.7341", (52.0976, 23.7341)), ("52.1;23.7", (52.1, 23.7)),
                                           ("", None), ("abc", None), ("95, 10", None)])
def test_parse_coords(text, expected):
    assert parse_coords(text) == expected


def test_journal_roundtrip(tmp_path):
    j = Journal(tmp_path / "logs" / "j.csv")
    j.add([fr().record("кадр")])
    j.add([fr(level=4, gps=(52.1, 23.7)).record("пакет"), fr(level=0, pothole=0, crack=0, n_pot=0).record("пакет")])
    df = j.load()
    assert len(df) == 3 and set(df["источник"]) == {"кадр", "пакет"}
    assert df["широта"].notna().sum() == 1
    j.clear()
    assert j.load().empty


def test_before_after_repaired():
    v = compare_before_after(fr(pothole=1.0, crack=3.0), fr(level=0, pothole=0, crack=0, n_pot=0))
    assert v["verdict"] == "ремонт выполнен"


def test_before_after_partial():
    v = compare_before_after(fr(pothole=1.0, crack=3.0, n_pot=2), fr(pothole=0.4, crack=1.0, n_pot=1))
    assert v["verdict"] == "ремонт выполнен частично" and v["ratio"] == pytest.approx(0.35)


def test_before_after_not_done():
    v = compare_before_after(fr(pothole=1.0, crack=3.0), fr(pothole=1.2, crack=3.5))
    assert v["verdict"] == "ремонт не выполнен"


def test_before_after_nothing_before():
    v = compare_before_after(fr(level=0, pothole=0, crack=0, n_pot=0), fr())
    assert v["verdict"] == "сравнение не требуется"


def test_sample_video(tmp_path):
    import cv2
    p = tmp_path / "v.mp4"
    w = cv2.VideoWriter(str(p), cv2.VideoWriter_fourcc(*"mp4v"), 10, (64, 48))
    for i in range(50):
        w.write(np.full((48, 64, 3), i * 5, np.uint8))
    w.release()
    frames = list(sample_video(p, step_s=1.0))
    assert len(frames) == 5 and frames[1][0] == pytest.approx(1.0) and frames[0][1].shape == (48, 64, 3)
    assert len(list(sample_video(p, step_s=0.5, max_frames=3))) == 3


def test_sample_video_bad_file(tmp_path):
    p = tmp_path / "bad.mp4"
    p.write_text("not a video")
    with pytest.raises(ValueError):
        list(sample_video(p))


def test_pdf_report(tmp_path):
    out = pdf_report([fr(gps=(52.1, 23.7)), fr(level=0, pothole=0, crack=0, n_pot=0, name="b.jpg")], tmp_path / "r.pdf")
    data = Path(out).read_bytes()
    assert data[:4] == b"%PDF" and data.count(b"/Type /Page") >= 2


def test_map_html():
    assert "Нет кадров" in map_html([])
    h = map_html([{"lat": 52.1, "lon": 23.7, "state": "аварийное", "level": 4, "name": "k", "time": ""}])
    assert "leaflet" in h and "52.1" in h


In [ ]:
%%writefile configs/knowledge_base.json
{
  "description": "База знаний экспертной оценки состояния дорожного покрытия по маске сегментации. Пороги — эвристические, задаются экспертом и могут меняться без правки кода. Факты в процентах — доля площади кадра.",
  "params": {
    "crack_minor_pct": 1.0,
    "crack_major_pct": 8.0,
    "pothole_large_pct": 2.5,
    "pothole_many": 3,
    "pothole_near_min_pct": 0.3,
    "near_zone_from": 0.6,
    "conf_min": 0.6
  },
  "param_descriptions": {
    "crack_minor_pct": "площадь трещин, % кадра, начиная с которой участок считается растрескавшимся",
    "crack_major_pct": "площадь трещин, % кадра, при которой нужен плановый ремонт",
    "pothole_large_pct": "площадь одной выбоины, % кадра, при которой она считается крупной",
    "pothole_many": "число выбоин в кадре, при котором участок считается аварийным",
    "pothole_near_min_pct": "мин. площадь выбоины в ближней зоне для предупреждения водителя, % кадра",
    "near_zone_from": "граница ближней зоны: доля высоты кадра сверху (0,6 — нижние 40 %)",
    "conf_min": "минимальная средняя уверенность модели по пикселям дефектов"
  },
  "severity_levels": {
    "0": {
      "name": "дефекты не обнаружены",
      "priority": "—"
    },
    "1": {
      "name": "хорошее",
      "priority": "низкий"
    },
    "2": {
      "name": "удовлетворительное",
      "priority": "плановый осмотр"
    },
    "3": {
      "name": "неудовлетворительное",
      "priority": "плановый ремонт"
    },
    "4": {
      "name": "аварийное",
      "priority": "срочный ремонт"
    }
  },
  "rules": [
    {
      "id": "D1",
      "name": "Наличие выбоин",
      "if": [
        [
          "pothole_count",
          ">",
          0
        ]
      ],
      "then": {
        "facts": {
          "has_pothole": true
        }
      }
    },
    {
      "id": "D2",
      "name": "Наличие зоны трещиноватости",
      "if": [
        [
          "crack_area_pct",
          ">=",
          "$crack_minor_pct"
        ]
      ],
      "then": {
        "facts": {
          "has_crack_zone": true
        }
      }
    },
    {
      "id": "D3",
      "name": "Прогрессирующее разрушение",
      "if": [
        [
          "has_pothole",
          "==",
          true
        ],
        [
          "has_crack_zone",
          "==",
          true
        ]
      ],
      "then": {
        "facts": {
          "progressive_damage": true
        },
        "message": "Выбоины расположены на участке с сеткой трещин — разрушение покрытия прогрессирует."
      }
    },
    {
      "id": "R0",
      "name": "Нет дефектов",
      "if": [
        [
          "pothole_count",
          "==",
          0
        ],
        [
          "crack_count",
          "==",
          0
        ]
      ],
      "then": {
        "severity": 0,
        "message": "Выбоины и трещины не обнаружены."
      }
    },
    {
      "id": "R1",
      "name": "Единичные мелкие трещины",
      "if": [
        [
          "has_pothole",
          "!=",
          true
        ],
        [
          "crack_count",
          ">",
          0
        ],
        [
          "crack_area_pct",
          "<",
          "$crack_minor_pct"
        ]
      ],
      "then": {
        "severity": 1,
        "message": "Обнаружены единичные трещины малой площади.",
        "recommendation": "Наблюдение при следующем плановом проезде."
      }
    },
    {
      "id": "R2",
      "name": "Зона трещиноватости средней площади",
      "if": [
        [
          "has_pothole",
          "!=",
          true
        ],
        [
          "crack_area_pct",
          ">=",
          "$crack_minor_pct"
        ],
        [
          "crack_area_pct",
          "<",
          "$crack_major_pct"
        ]
      ],
      "then": {
        "severity": 2,
        "message": "Покрытие на участке растрескалось.",
        "recommendation": "Герметизация (заливка) трещин, повторный осмотр."
      }
    },
    {
      "id": "R3",
      "name": "Обширная зона трещиноватости",
      "if": [
        [
          "crack_area_pct",
          ">=",
          "$crack_major_pct"
        ]
      ],
      "then": {
        "severity": 3,
        "message": "Трещины покрывают значительную часть кадра.",
        "recommendation": "Плановый ремонт покрытия участка (фрезерование, укладка слоя)."
      }
    },
    {
      "id": "R4",
      "name": "Выбоины умеренного размера",
      "if": [
        [
          "has_pothole",
          "==",
          true
        ],
        [
          "pothole_max_pct",
          "<",
          "$pothole_large_pct"
        ],
        [
          "pothole_count",
          "<",
          "$pothole_many"
        ]
      ],
      "then": {
        "severity": 3,
        "message": "Обнаружены выбоины.",
        "recommendation": "Ямочный ремонт в плановом порядке."
      }
    },
    {
      "id": "R5",
      "name": "Крупная выбоина",
      "if": [
        [
          "pothole_max_pct",
          ">=",
          "$pothole_large_pct"
        ]
      ],
      "then": {
        "severity": 4,
        "message": "Обнаружена крупная выбоина.",
        "recommendation": "Срочный ямочный ремонт; до устранения — ограждение и временные знаки."
      }
    },
    {
      "id": "R6",
      "name": "Множественные выбоины",
      "if": [
        [
          "pothole_count",
          ">=",
          "$pothole_many"
        ]
      ],
      "then": {
        "severity": 4,
        "message": "Обнаружено много выбоин на одном участке.",
        "recommendation": "Срочный ремонт участка."
      }
    },
    {
      "id": "R7",
      "name": "Выбоины на участке с сеткой трещин",
      "if": [
        [
          "progressive_damage",
          "==",
          true
        ]
      ],
      "then": {
        "severity": 3,
        "recommendation": "Ремонтировать участок целиком, а не только отдельные выбоины."
      }
    },
    {
      "id": "R7a",
      "name": "Выбоины на обширной зоне трещин",
      "if": [
        [
          "progressive_damage",
          "==",
          true
        ],
        [
          "crack_area_pct",
          ">=",
          "$crack_major_pct"
        ]
      ],
      "then": {
        "severity": 4,
        "message": "Выбоины на обширной зоне трещиноватости — покрытие разрушено.",
        "recommendation": "Срочный ремонт участка."
      }
    },
    {
      "id": "R8",
      "name": "Выбоина непосредственно перед ТС",
      "if": [
        [
          "pothole_near_max_pct",
          ">=",
          "$pothole_near_min_pct"
        ]
      ],
      "then": {
        "facts": {
          "driver_warning": true
        },
        "message": "Выбоина в ближней зоне перед автомобилем — предупреждение водителю."
      }
    },
    {
      "id": "R9",
      "name": "Люк — не дефект",
      "if": [
        [
          "manhole_count",
          ">",
          0
        ]
      ],
      "then": {
        "message": "Обнаружен люк: не является дефектом, учитывается как элемент обустройства.",
        "recommendation": "Проверить отсутствие просадки покрытия вокруг люка."
      }
    },
    {
      "id": "R10",
      "name": "Низкая уверенность модели",
      "if": [
        [
          "mean_conf",
          "<",
          "$conf_min"
        ],
        [
          "defect_area_pct",
          ">",
          0
        ]
      ],
      "then": {
        "facts": {
          "needs_review": true
        },
        "message": "Уверенность модели низкая — результат нужно проверить оператору."
      }
    }
  ]
}

In [ ]:
%%writefile tests/test_core.py
import sys
from pathlib import Path

import numpy as np
import pytest
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import CLASS_NAMES, NUM_CLASSES
from src.data import find_pairs, make_split
from src.labels import parse_line, read_label_file
from src.masks import compose_semantic_mask, polygon_mask
from src.metrics import SegMetrics
from src.model import DiceCELoss, build_model, compute_class_weights


def test_parse_obb_line():
    a = parse_line("1 0.1 0.2 0.5 0.2 0.5 0.6 0.1 0.6", 100, 50)
    assert a.cls_id == 1
    assert a.polygon.shape == (4, 2)
    assert a.bbox == pytest.approx((10, 10, 50, 30))
    assert a.area == pytest.approx(40 * 20)


def test_parse_yolo_box_line():
    a = parse_line("0 0.5 0.5 0.2 0.4", 200, 100)
    assert a.bbox == pytest.approx((80, 30, 120, 70))


def test_parse_polygon_line():
    a = parse_line("2 0.1 0.1 0.3 0.1 0.3 0.3 0.2 0.4 0.1 0.3", 100, 100)
    assert a.polygon.shape == (5, 2)


@pytest.mark.parametrize("line", ["", "abc", "1 0.1 0.2", "0 0.5 0.5 0 0"])
def test_parse_bad_lines(line):
    assert parse_line(line, 100, 100) is None


def test_coords_are_clipped():
    a = parse_line("0 -0.2 -0.2 1.5 -0.2 1.5 1.5 -0.2 1.5", 10, 10)
    assert a.polygon.min() >= 0 and a.polygon.max() <= 10


def test_read_label_file_filters_classes(tmp_path):
    f = tmp_path / "a.txt"
    f.write_text("0 0.1 0.1 0.2 0.1 0.2 0.2 0.1 0.2\n7 0.1 0.1 0.2 0.1 0.2 0.2 0.1 0.2\nbroken\n")
    anns, skipped = read_label_file(f, 100, 100, {0, 1, 2})
    assert len(anns) == 1 and skipped == 2


def test_polygon_mask_area():
    a = parse_line("0 0.1 0.1 0.5 0.1 0.5 0.5 0.1 0.5", 100, 100)
    m = polygon_mask(a, 100, 100)
    assert abs(int(m.sum()) - 41 * 41) <= 82


def test_small_objects_drawn_on_top():
    big = np.zeros((20, 20), bool); big[:, :] = True
    small = np.zeros((20, 20), bool); small[5:8, 5:8] = True
    sem = compose_semantic_mask([(3, big), (2, small)], 20, 20)
    assert sem[6, 6] == 2 and sem[0, 0] == 3


def test_make_split_proportions():
    s = make_split(100, 0.15, 0.15, 0)
    assert s.count("test") == 15 and s.count("val") == 15 and s.count("train") == 70
    assert s == make_split(100, 0.15, 0.15, 0)


def test_find_pairs(tmp_path):
    (tmp_path / "images").mkdir(); (tmp_path / "labels").mkdir()
    from PIL import Image
    Image.new("RGB", (8, 8)).save(tmp_path / "images" / "x.jpg")
    Image.new("RGB", (8, 8)).save(tmp_path / "images" / "y.jpg")
    (tmp_path / "labels" / "x.txt").write_text("0 0.5 0.5 0.2 0.2")
    (tmp_path / "labels" / "classes.txt").write_text("pothole")
    pairs = dict((p.name, l) for p, l in find_pairs(tmp_path))
    assert pairs["x.jpg"].name == "x.txt" and pairs["y.jpg"] is None


def test_metrics_perfect():
    t = np.random.randint(0, NUM_CLASSES, (2, 16, 16))
    m = SegMetrics(NUM_CLASSES, CLASS_NAMES)
    m.update(t, t)
    s = m.summary()
    assert s["mIoU"] == pytest.approx(1.0) and s["pixel_accuracy"] == pytest.approx(1.0)


def test_metrics_known_values():
    gt = np.array([[1, 1, 0, 0]])
    pr = np.array([[1, 0, 1, 0]])
    m = SegMetrics(NUM_CLASSES, CLASS_NAMES)
    m.update(pr, gt)
    pc = m.per_class()["pothole"]
    assert pc["iou"] == pytest.approx(1 / 3)
    assert pc["dice"] == pytest.approx(0.5)
    assert pc["precision"] == pytest.approx(0.5)
    assert pc["recall"] == pytest.approx(0.5)
    assert np.isnan(m.per_class()["crack"]["iou"])
    assert m.summary()["mIoU"] == pytest.approx(1 / 3)


def test_binary_merges_defect_classes():
    gt = np.array([[1, 2, 0]])
    pr = np.array([[2, 1, 0]])
    m = SegMetrics(NUM_CLASSES, CLASS_NAMES)
    m.update(pr, gt)
    assert m.binary()["iou"] == pytest.approx(1.0)
    assert m.per_class()["pothole"]["iou"] == pytest.approx(0.0)


def test_class_weights_monotonic():
    w = compute_class_weights([1_000_000, 10_000, 1_000, 100_000])
    assert w[0] == pytest.approx(1.0) and w[2] >= w[1] >= w[3] > 1


def test_loss_decreases_for_correct_prediction():
    y = torch.randint(0, NUM_CLASSES, (2, 8, 8))
    good = torch.nn.functional.one_hot(y, NUM_CLASSES).permute(0, 3, 1, 2).float() * 10
    bad = torch.randn(2, NUM_CLASSES, 8, 8)
    loss = DiceCELoss(NUM_CLASSES)
    assert loss(good, y) < loss(bad, y)


@pytest.mark.parametrize("arch", ["unet", "deeplabv3plus", "fpn"])
def test_model_output_shape(arch):
    m = build_model(arch, "resnet18", NUM_CLASSES, encoder_weights=None).eval()
    with torch.no_grad():
        out = m(torch.randn(1, 3, 64, 128))
    assert out.shape == (1, NUM_CLASSES, 64, 128)


In [ ]:
%%writefile tests/test_knowledge.py
import json
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.knowledge import DEFAULT_KB, InferenceEngine, KnowledgeBase, assess, extract_facts

H, W = 100, 200


@pytest.fixture(scope="module")
def kb():
    return KnowledgeBase()


def blank():
    return np.zeros((H, W), np.uint8)


def ids(res):
    return {f["id"] for f in res.fired}


def test_kb_loads_and_validates(kb):
    assert len(kb.rules) >= 10 and "crack_major_pct" in kb.params


def test_bad_kb_rejected(tmp_path):
    data = json.loads(DEFAULT_KB.read_text(encoding="utf-8"))
    data["rules"][0]["if"][0][2] = "$нет_такого"
    p = tmp_path / "kb.json"
    p.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    with pytest.raises(ValueError):
        KnowledgeBase(p)


def test_facts_extraction():
    m = blank()
    m[10:20, 10:30] = 1
    m[80:90, 100:120] = 1
    m[40:60, 50:150] = 2
    f = extract_facts(m)
    assert f["pothole_count"] == 2 and f["pothole_area_pct"] == pytest.approx(2.0)
    assert f["pothole_max_pct"] == pytest.approx(1.0)
    assert f["pothole_near_max_pct"] == pytest.approx(1.0)
    assert f["crack_area_pct"] == pytest.approx(10.0)
    assert f["manhole_count"] == 0


def test_no_defects(kb):
    r = assess(blank(), kb=kb)
    assert r.level == 0 and "R0" in ids(r)


def test_manhole_is_not_defect(kb):
    m = blank(); m[10:30, 10:30] = 3
    r = assess(m, kb=kb)
    assert r.level == 0 and {"R0", "R9"} <= ids(r)


def test_medium_cracks(kb):
    m = blank(); m[10:30, 10:30] = 2
    r = assess(m, kb=kb)
    assert r.level == 2 and "R2" in ids(r)


def test_large_crack_zone(kb):
    m = blank(); m[0:50, 0:40] = 2
    r = assess(m, kb=kb)
    assert r.level == 3 and "R3" in ids(r)


def test_small_pothole(kb):
    m = blank(); m[10:20, 10:20] = 1
    r = assess(m, kb=kb)
    assert r.level == 3 and "R4" in ids(r) and not r.facts.get("driver_warning")


def test_large_pothole_is_emergency(kb):
    m = blank(); m[10:40, 10:30] = 1
    r = assess(m, kb=kb)
    assert r.level == 4 and "R5" in ids(r)


def test_many_potholes(kb):
    m = blank()
    for x in (10, 60, 110):
        m[10:15, x:x + 10] = 1
    r = assess(m, kb=kb)
    assert r.level == 4 and "R6" in ids(r)


def test_forward_chaining_progressive_damage(kb):
    m = blank()
    m[10:30, 10:30] = 2
    m[15:20, 15:20] = 1
    r = assess(m, kb=kb)
    assert {"D1", "D2", "D3", "R7"} <= ids(r)
    assert r.facts["progressive_damage"] is True
    order = [f["id"] for f in r.fired]
    assert order.index("D3") > order.index("D1") and order.index("D3") > order.index("D2")


def test_driver_warning(kb):
    m = blank(); m[85:95, 90:110] = 1
    r = assess(m, kb=kb)
    assert r.facts.get("driver_warning") is True and "R8" in ids(r)


def test_low_confidence_flag(kb):
    m = blank(); m[10:30, 10:30] = 2
    conf = np.full((H, W), 0.4, np.float32)
    r = assess(m, conf, kb=kb)
    assert r.facts.get("needs_review") is True


def test_each_rule_fires_once(kb):
    m = blank(); m[0:50, 0:40] = 2; m[5:10, 5:10] = 1
    r = InferenceEngine(kb).run(extract_facts(m))
    order = [f["id"] for f in r.fired]
    assert len(order) == len(set(order))


def test_explanation_mentions_facts(kb):
    m = blank(); m[10:40, 10:30] = 1
    text = assess(m, kb=kb).explanation()
    assert "R5" in text and "pothole_max_pct" in text


In [ ]:
%%writefile app.py
from __future__ import annotations

import argparse
import tempfile
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from src.config import CLASS_COLORS, CLASS_NAMES, CLASS_NAMES_RU, DEFECT_CLASS_IDS, NUM_CLASSES
from src.inference import load_model, predict_mask, remove_small_regions
from src.knowledge import InferenceEngine, KnowledgeBase, extract_facts
from src.masks import colorize, overlay
from src.metrics import SegMetrics
from src.monitoring import (LEVEL_COLORS, FrameResult, Journal, compare_before_after, map_html, parse_coords,
                            pdf_report, read_gps, sample_video, video_timeline_png)

ACCENT = "#F5EE0A"
INK = "#111111"
OUT_DIR = Path(tempfile.gettempdir()) / "road_defect_app"
OUT_DIR.mkdir(exist_ok=True)


def parse_args():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ckpt", required=True, help="веса модели (best.pt)")
    ap.add_argument("--data", default="prepared", help="папка с index.csv — для примеров и эталонных масок")
    ap.add_argument("--split", default="test")
    ap.add_argument("--examples-dir", default=None, help="папка с кадрами-примерами (вместо выборки из index.csv)")
    ap.add_argument("--n-examples", type=int, default=24)
    ap.add_argument("--kb", default=None, help="файл базы знаний (по умолчанию configs/knowledge_base.json)")
    ap.add_argument("--journal", default="logs/journal.csv", help="файл журнала проверок")
    ap.add_argument("--share", action="store_true", help="создать публичную ссылку gradio.live")
    ap.add_argument("--port", type=int, default=7860)
    ap.add_argument("--inbrowser", action="store_true", help="сразу открыть интерфейс в браузере")
    return ap.parse_args()


def confidence_map(rgb: np.ndarray, conf: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB).astype(np.float32) * 0.55 + 90
    unc = np.clip((1.0 - conf) / 0.5, 0, 1)[..., None]
    heat = np.array([245, 238, 10], np.float32) * (1 - unc) + np.array([224, 36, 36], np.float32) * unc
    a = np.clip(unc * 1.6, 0, 0.85)
    return (gray * (1 - a) + heat * a).clip(0, 255).astype(np.uint8)


def _path(f):
    return f if isinstance(f, str) else getattr(f, "name", None) or getattr(f, "path", None)


class Segmenter:
    def __init__(self, ckpt: str, data: str, split: str, kb_path: str | None = None,
                 journal: str = "logs/journal.csv"):
        self.model, self.meta, self.device = load_model(ckpt)
        self.kb = KnowledgeBase(kb_path) if kb_path else KnowledgeBase()
        self.engine = InferenceEngine(self.kb)
        self.journal = Journal(journal)
        self.gt_by_image: dict[str, Path] = {}
        self.examples: list[str] = []
        index = Path(data) / "index.csv"
        if index.exists():
            df = pd.read_csv(index)
            for _, r in df.iterrows():
                self.gt_by_image[Path(r["image_path"]).name] = Path(data) / r["mask_path"]
            sub = df[df["split"] == split]
            sub = sub.assign(_k=(sub.filter(like="n_") > 0).sum(axis=1)).sort_values("_k", ascending=False)
            self.examples = [p for p in sub["image_path"] if Path(p).exists()]

    def analyze(self, rgb: np.ndarray, name: str, min_area: int = 30, tta: bool = False, alpha: float = 0.55,
                gps=None, time_s=None):
        mask, conf = predict_mask(self.model, self.meta, self.device, rgb, tta)
        mask, regions = remove_small_regions(mask, int(min_area), DEFECT_CLASS_IDS)
        res = self.engine.run(extract_facts(mask, conf, self.kb.params.get("near_zone_from", 0.6)))
        fr = FrameResult(
            name=name, overlay=overlay(rgb, mask, CLASS_COLORS, alpha), mask=mask, level=res.level,
            state=res.level_name, priority=res.priority,
            areas={CLASS_NAMES[c]: round(100 * float((mask == c).mean()), 2) for c in DEFECT_CLASS_IDS},
            regions={CLASS_NAMES[c]: regions[c] for c in DEFECT_CLASS_IDS},
            messages=res.messages, recommendations=res.recommendations, rules=[f["id"] for f in res.fired],
            gps=gps, time_s=time_s)
        return fr, res, conf

    @staticmethod
    def _read(path):
        bgr = cv2.imread(path) if path else None
        return None if bgr is None else cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    def segment(self, image_path, alpha, min_area, tta, coords_text):
        import gradio as gr
        empty = (None, None, gr.update(value=None, visible=False), pd.DataFrame(), "", "", "", None)
        if not image_path:
            return empty[:4] + ("Загрузите изображение или выберите кадр из списка.",) + empty[5:]
        rgb = self._read(image_path)
        if rgb is None:
            return empty[:4] + ("⚠️ Файл не удалось прочитать как изображение. Поддерживаются JPG, PNG, BMP.",) + empty[5:]
        gps = parse_coords(coords_text) or read_gps(image_path)
        t0 = time.perf_counter()
        fr, res, conf = self.analyze(rgb, Path(image_path).name, min_area, tta, alpha, gps)
        dt = (time.perf_counter() - t0) * 1000
        self.journal.add([fr.record("кадр")])

        rows = []
        for c in DEFECT_CLASS_IDS:
            sel = fr.mask == c
            rows.append({"Класс": CLASS_NAMES_RU[c], "Областей": fr.regions[CLASS_NAMES[c]],
                         "Площадь, % кадра": fr.areas[CLASS_NAMES[c]],
                         "Ср. уверенность": round(float(conf[sel].mean()), 3) if sel.any() else None})
        found = [f"{r['Класс']} ({r['Областей']})" for r in rows if r["Областей"]]
        md = [f"**Обнаружено:** {', '.join(found)}" if found else "**Дефекты не обнаружены.**",
              f"Время обработки: {dt:.0f} мс · {self.meta['arch']}/{self.meta['encoder']} · {self.device.type.upper()}",
              f"Координаты: {gps[0]:.6f}, {gps[1]:.6f}" if gps else "Координаты: нет (можно указать вручную)"]
        gt_vis = None
        gt_path = self.gt_by_image.get(Path(image_path).name)
        if gt_path is not None and gt_path.exists():
            gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
            gt_vis = overlay(rgb, gt, CLASS_COLORS, alpha)
            m = SegMetrics(NUM_CLASSES, CLASS_NAMES, DEFECT_CLASS_IDS)
            m.update(fr.mask, gt)
            b = m.binary()
            md.append(f"Сравнение с разметкой: IoU «дефект/фон» = **{b['iou']:.3f}**, "
                      f"точность = {b['precision']:.3f}, полнота = {b['recall']:.3f}")
        report = pdf_report([fr], OUT_DIR / f"заявка_{Path(image_path).stem}.pdf", "Заявка на обследование участка")
        import gradio as gr
        return (fr.overlay, confidence_map(rgb, conf), gr.update(value=gt_vis, visible=gt_vis is not None),
                pd.DataFrame(rows), "\n\n".join(md), self.format_assessment(res), res.explanation(), report)

    def format_assessment(self, res) -> str:
        color = LEVEL_COLORS.get(res.level, "#888")
        out = [f"<div class='state' style='border-color:{color}'><span class='dot' style='background:{color}'></span>"
               f"<b>{res.level_name.upper()}</b><span class='prio'>приоритет: {res.priority}</span></div>"]
        if res.facts.get("driver_warning"):
            out.append("<p class='warn'>▲ Выбоина в ближней зоне перед автомобилем</p>")
        if res.messages:
            out.append("<p class='h'>Выводы</p><ul>" + "".join(f"<li>{m}</li>" for m in res.messages) + "</ul>")
        if res.recommendations:
            out.append("<p class='h'>Рекомендации</p><ul>" + "".join(f"<li>{m}</li>" for m in res.recommendations) + "</ul>")
        return "".join(out)

    def batch(self, files, min_area, tta):
        if not files:
            return [], pd.DataFrame(), None, None, map_html([])
        gallery, rows, results, records = [], [], [], []
        for f in files:
            path = _path(f)
            rgb = self._read(path)
            if rgb is None:
                rows.append({"Кадр": Path(path).name, "Состояние": "ошибка: файл не является изображением"})
                continue
            fr, res, _ = self.analyze(rgb, Path(path).name, min_area, tta, gps=read_gps(path))
            results.append(fr)
            rec = fr.record("пакет")
            records.append(rec)
            rows.append({k: v for k, v in rec.items() if k not in ("время", "источник", "уровень", "рекомендации")})
            gallery.append((fr.overlay, f"{fr.name} — {fr.state}"))
        self.journal.add(records)
        df = pd.DataFrame(rows).rename(columns={"кадр": "Кадр", "состояние": "Состояние"})
        csv_path = OUT_DIR / "пакет.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        pdf_path = pdf_report(results, OUT_DIR / "отчет_пакет.pdf") if results else None
        pts = [{"lat": r.gps[0], "lon": r.gps[1], "state": r.state, "level": r.level, "name": r.name, "time": ""}
               for r in results if r.gps]
        return gallery, df, str(csv_path), pdf_path, map_html(pts, 420)

    def video(self, file, step_s, max_frames, min_area, progress=None):
        path = _path(file)
        if not path:
            return None, [], pd.DataFrame(), None, None
        results = []
        try:
            for t, rgb in sample_video(path, float(step_s), int(max_frames)):
                fr, _, _ = self.analyze(rgb, f"{Path(path).stem} @ {t:.1f} с", min_area, time_s=round(t, 2))
                results.append(fr)
        except ValueError as e:
            raise RuntimeError(f"Видео не удалось прочитать: {e}")
        if not results:
            raise RuntimeError("В видео не найдено ни одного кадра")
        self.journal.add([r.record("видео") for r in results])
        png = OUT_DIR / "видео_график.png"
        video_timeline_png(results, png)
        worst = sorted(results, key=lambda r: (-r.level, -r.defect_area))[:6]
        gallery = [(r.overlay, f"{r.time_s:.1f} с — {r.state}") for r in worst]
        df = pd.DataFrame([{"время, с": r.time_s, "состояние": r.state, "приоритет": r.priority,
                            "выбоины, %": r.areas["pothole"], "трещины, %": r.areas["crack"],
                            "люки": r.regions["manhole"], "правила": ", ".join(r.rules)} for r in results])
        csv_path = OUT_DIR / "видео.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        pdf_path = pdf_report(results, OUT_DIR / "отчет_видео.pdf", "Отчет по видеозаписи участка")
        return str(png), gallery, df, str(csv_path), pdf_path

    def before_after(self, before_path, after_path, ratio_pct, min_area):
        if not before_path or not after_path:
            return None, None, "<p>Загрузите оба снимка участка.</p>", pd.DataFrame()
        a, b = self._read(before_path), self._read(after_path)
        if a is None or b is None:
            return None, None, "<p>⚠️ Один из файлов не является изображением.</p>", pd.DataFrame()
        fa, _, _ = self.analyze(a, "до", min_area)
        fb, _, _ = self.analyze(b, "после", min_area)
        v = compare_before_after(fa, fb, float(ratio_pct) / 100)
        color = {"ремонт выполнен": "#4ade80", "ремонт выполнен частично": "#facc15",
                 "ремонт не выполнен": "#ef4444"}.get(v["verdict"], "#9ca3af")
        html = (f"<div class='state' style='border-color:{color}'><span class='dot' style='background:{color}'></span>"
                f"<b>{v['verdict'].upper()}</b></div><p>{v['detail']}</p>")
        df = pd.DataFrame([{"Снимок": "до", "Состояние": fa.state, "Выбоины, %": fa.areas["pothole"],
                            "Трещины, %": fa.areas["crack"], "Выбоин": fa.regions["pothole"]},
                           {"Снимок": "после", "Состояние": fb.state, "Выбоины, %": fb.areas["pothole"],
                            "Трещины, %": fb.areas["crack"], "Выбоин": fb.regions["pothole"]}])
        return fa.overlay, fb.overlay, html, df

    def journal_view(self, state_filter):
        df = self.journal.load()
        if df.empty:
            return df, None
        if state_filter and state_filter != "все":
            df = df[df["состояние"] == state_filter]
        df = df.iloc[::-1].reset_index(drop=True)
        path = OUT_DIR / "журнал.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return df, str(path)

    def journal_clear(self):
        self.journal.clear()
        return pd.DataFrame(), None

    def map_view(self):
        df = self.journal.load()
        if df.empty or "широта" not in df:
            return map_html([]), "Записей с координатами: 0"
        df = df[pd.to_numeric(df["широта"], errors="coerce").notna()]
        df = df.drop_duplicates(subset=["кадр", "широта", "долгота"], keep="last")
        pts = [{"lat": float(r["широта"]), "lon": float(r["долгота"]), "state": r["состояние"],
                "level": int(r["уровень"]), "name": str(r["кадр"]), "time": str(r["время"])} for _, r in df.iterrows()]
        return map_html(pts), f"Записей с координатами: {len(pts)}"


LOGO = """<div class="brand"><svg width="34" height="34" viewBox="0 0 40 40" xmlns="http://www.w3.org/2000/svg" aria-label="ROADSEG">
<path d="M5 37 L15.5 3 M35 37 L24.5 3" stroke="#111111" stroke-width="5" stroke-linecap="butt"/>
<path d="M20 6 V10 M20 15 V20 M20 25 V31" stroke="#111111" stroke-width="2.6"/>
<rect x="22.5" y="20.5" width="9" height="8" fill="#F5EE0A" stroke="#111111" stroke-width="2"/>
</svg><span class="word">ROADSEG</span></div>"""

CSS = """
.gradio-container { background:#FFFFFF !important; max-width:1360px !important; }
footer { display:none !important; }
#hdr { display:flex; align-items:center; gap:22px; padding:16px 4px 16px; border-bottom:1px solid #E6E6E6; margin-bottom:6px; }
#hdr .brand { display:flex; align-items:center; gap:10px; }
#hdr .word { font-size:25px; font-weight:800; letter-spacing:-.01em; color:#111111; line-height:1; }
#hdr .sep { width:1px; height:26px; background:#D9D9D9; }
#hdr .title { font-size:15px; color:#555555; }
#hdr .legend { margin-left:auto; color:#333333; font-size:13px; }
.legend span.sw { display:inline-block; width:10px; height:10px; border-radius:50%; margin:0 6px 0 16px; vertical-align:middle; }
button[role=tab] { font-size:14px; color:#666666 !important; }
button[role=tab][aria-selected=true] { color:#111111 !important; border-bottom:2px solid #111111 !important; font-weight:600; }
.block, .form { border-radius:2px !important; box-shadow:none !important; }
.state { display:flex; align-items:center; gap:12px; padding:12px 16px; border:1px solid #E6E6E6; border-left:5px solid;
  border-radius:2px; background:#FFFFFF; font-size:16px; color:#111111; }
.state b { color:#111111 !important; font-weight:700; }
.state .dot { display:none; }
.state .prio { margin-left:auto; color:#666666; font-size:14px; }
p.h { color:#111111; font-size:14px; font-weight:700; margin:14px 0 4px; }
p.warn { color:#E02424; font-weight:700; margin-top:10px; }
.prose, .md { padding-left:6px !important; }
table, table * { font-family:inherit !important; }
"""


def build_theme():
    import gradio as gr
    return gr.themes.Base(
        primary_hue=gr.themes.Color(c50="#fffde0", c100="#fffbb8", c200="#fff88a", c300="#fbf45a", c400="#f8f130",
                                    c500="#F5EE0A", c600="#d6cf00", c700="#a8a300", c800="#7a7600", c900="#4d4a00",
                                    c950="#2b2900"),
        neutral_hue="stone",
        font=[gr.themes.Font("Inter"), gr.themes.Font("Helvetica Neue"), gr.themes.Font("Arial"), gr.themes.Font("sans-serif")],
        font_mono=[gr.themes.Font("Inter"), gr.themes.Font("Helvetica Neue"), gr.themes.Font("Arial"), gr.themes.Font("sans-serif")],
        radius_size="sm",
    ).set(
        body_background_fill="#FFFFFF", body_background_fill_dark="#FFFFFF",
        body_text_color=INK, body_text_color_dark=INK,
        block_background_fill="#FFFFFF", block_background_fill_dark="#FFFFFF",
        block_border_color="#E6E6E6", block_border_color_dark="#E6E6E6",
        block_label_background_fill="#FFFFFF", block_label_background_fill_dark="#FFFFFF",
        block_label_text_color=INK, block_label_text_color_dark=INK,
        block_title_text_color=INK, block_title_text_color_dark=INK,
        input_background_fill="#FAFAFA", input_background_fill_dark="#FAFAFA",
        border_color_primary="#E6E6E6", border_color_primary_dark="#E6E6E6",
        button_primary_background_fill=ACCENT, button_primary_background_fill_dark=ACCENT,
        button_primary_background_fill_hover="#E6DF00", button_primary_background_fill_hover_dark="#E6DF00",
        button_primary_text_color=INK, button_primary_text_color_dark=INK,
        button_secondary_background_fill=INK, button_secondary_background_fill_dark=INK,
        button_secondary_background_fill_hover="#333333", button_secondary_background_fill_hover_dark="#333333",
        button_secondary_text_color="#FFFFFF", button_secondary_text_color_dark="#FFFFFF",
        button_large_radius="2px", button_small_radius="2px",
        slider_color=INK, slider_color_dark=INK,
        color_accent_soft="#FFF9B0", color_accent_soft_dark="#FFF9B0",
        table_even_background_fill="#FFFFFF", table_even_background_fill_dark="#FFFFFF",
        table_odd_background_fill="#FAFAFA", table_odd_background_fill_dark="#FAFAFA",
        background_fill_primary="#FFFFFF", background_fill_primary_dark="#FFFFFF",
        background_fill_secondary="#FAFAFA", background_fill_secondary_dark="#FAFAFA",
    )


def header_html() -> str:
    legend = "".join(f"<span class='sw' style='background:rgb{CLASS_COLORS[c]}'></span>{CLASS_NAMES_RU[c]}"
                     for c in DEFECT_CLASS_IDS)
    return (f"<div id='hdr'>{LOGO}<div class='sep'></div>"
            f"<div class='title'>Сегментация дефектов дорожного покрытия</div>"
            f"<div class='legend'>{legend}</div></div>")


def _gradio_major() -> int:
    import gradio as gr
    try:
        return int(gr.__version__.split(".")[0])
    except ValueError:
        return 5


def style_kwargs() -> dict:
    return {"theme": build_theme(), "css": CSS}


def build_ui(seg: Segmenter):
    import gradio as gr

    blocks_kw = {} if _gradio_major() >= 6 else style_kwargs()
    with gr.Blocks(title="Сегментация дефектов дорожного покрытия", **blocks_kw) as demo:
        gr.HTML(header_html())
        with gr.Tab("Кадр"):
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="filepath", label="Исходный кадр", height=280)
                    coords = gr.Textbox(label="Координаты (если в фото нет GPS)", placeholder="52.0976, 23.7341")
                    alpha = gr.Slider(0.1, 1.0, 0.55, step=0.05, label="Прозрачность маски")
                    min_area = gr.Slider(0, 500, 30, step=10, label="Мин. площадь области, пикс.")
                    tta = gr.Checkbox(False, label="TTA (усреднение с отраженным кадром)")
                    btn = gr.Button("Обновить с новыми настройками", variant="secondary")
                    report = gr.File(label="Заявка на обследование (PDF)")
                with gr.Column(scale=2):
                    out_overlay = gr.Image(label="Результат", height=360)
                    with gr.Row():
                        out_mask = gr.Image(label="Карта уверенности модели", height=190)
                        out_gt = gr.Image(label="Эталонная разметка", height=190, visible=False)
                    out_assess = gr.HTML()
                    with gr.Accordion("Объяснение вывода: сработавшие правила базы знаний", open=False):
                        out_expl = gr.Textbox(lines=8, show_label=False)
                    out_md = gr.Markdown()
                    out_table = gr.Dataframe(label="Сводка по классам", interactive=False)
            outs = [out_overlay, out_mask, out_gt, out_table, out_md, out_assess, out_expl, report]
            args = [inp, alpha, min_area, tta, coords]
            btn.click(seg.segment, args, outs)
            inp.change(seg.segment, args, outs)
            if seg.examples:
                gr.Examples(examples=[[p] for p in seg.examples], inputs=[inp], label="Кадры для проверки",
                            examples_per_page=12)
        with gr.Tab("Пакет"):
            files = gr.File(file_count="multiple", file_types=["image"], label="Кадры маршрута")
            with gr.Row():
                b_min = gr.Slider(0, 500, 30, step=10, label="Мин. площадь области, пикс.")
                b_tta = gr.Checkbox(False, label="TTA")
                b_btn = gr.Button("Обработать", variant="primary")
            gal = gr.Gallery(label="Результаты", columns=4, height="auto")
            b_table = gr.Dataframe(label="Сводка по кадрам", interactive=False)
            with gr.Row():
                b_csv = gr.File(label="Таблица (CSV)")
                b_pdf = gr.File(label="Отчет (PDF)")
            b_map = gr.HTML()
            b_btn.click(seg.batch, [files, b_min, b_tta], [gal, b_table, b_csv, b_pdf, b_map])
        with gr.Tab("Видео"):
            vfile = gr.File(file_types=["video"], label="Видео с регистратора (MP4, AVI, MOV)")
            with gr.Row():
                v_step = gr.Slider(0.2, 5, 1.0, step=0.2, label="Шаг между кадрами, с")
                v_max = gr.Slider(10, 600, 120, step=10, label="Не более кадров")
                v_min = gr.Slider(0, 500, 30, step=10, label="Мин. площадь области, пикс.")
                v_btn = gr.Button("Разобрать видео", variant="primary")
            v_plot = gr.Image(label="Состояние покрытия по времени", height=300)
            v_gal = gr.Gallery(label="Самые проблемные моменты", columns=3, height="auto")
            v_table = gr.Dataframe(label="Кадры", interactive=False, max_height=320)
            with gr.Row():
                v_csv = gr.File(label="Таблица (CSV)")
                v_pdf = gr.File(label="Отчет (PDF)")
            v_btn.click(seg.video, [vfile, v_step, v_max, v_min], [v_plot, v_gal, v_table, v_csv, v_pdf])
        with gr.Tab("До / после"):
            gr.Markdown("Загрузите два снимка одного участка: до ремонта и после. Система сравнит площадь выбоин и "
                        "трещин и сделает вывод о выполнении ремонта. Снимки желательно делать с одной точки.")
            with gr.Row():
                im_b = gr.Image(type="filepath", label="До ремонта", height=240)
                im_a = gr.Image(type="filepath", label="После ремонта", height=240)
            with gr.Row():
                ba_ratio = gr.Slider(5, 80, 30, step=5, label="Остаток площади дефектов, при котором ремонт засчитан, %")
                ba_min = gr.Slider(0, 500, 30, step=10, label="Мин. площадь области, пикс.")
                ba_btn = gr.Button("Сравнить", variant="primary")
            with gr.Row():
                ba_o1 = gr.Image(label="До", height=260)
                ba_o2 = gr.Image(label="После", height=260)
            ba_html = gr.HTML()
            ba_table = gr.Dataframe(interactive=False)
            ba_btn.click(seg.before_after, [im_b, im_a, ba_ratio, ba_min], [ba_o1, ba_o2, ba_html, ba_table])
        with gr.Tab("Карта"):
            with gr.Row():
                m_btn = gr.Button("Обновить карту", variant="primary", scale=0)
                m_info = gr.Markdown()
            m_html = gr.HTML()
            m_btn.click(seg.map_view, None, [m_html, m_info])
        with gr.Tab("Журнал"):
            with gr.Row():
                j_filter = gr.Dropdown(["все"] + [seg.kb.levels[k]["name"] for k in sorted(seg.kb.levels)],
                                       value="все", label="Состояние")
                j_btn = gr.Button("Показать", variant="primary")
                j_clear = gr.Button("Очистить журнал", variant="stop")
            j_table = gr.Dataframe(interactive=False, max_height=520)
            j_csv = gr.File(label="Журнал (CSV)")
            j_btn.click(seg.journal_view, [j_filter], [j_table, j_csv])
            j_clear.click(seg.journal_clear, None, [j_table, j_csv])
        with gr.Tab("База знаний"):
            gr.Markdown("Пороги и продукционные правила хранятся в `configs/knowledge_base.json` и меняются без "
                        "правки кода. Машина вывода применяет правила прямым выводом до неподвижной точки; итоговое "
                        "состояние — максимальный уровень сработавших правил.")
            desc = seg.kb.param_descriptions
            gr.Dataframe(pd.DataFrame([{"Параметр": k, "Значение": v, "Смысл": desc.get(k, "")}
                                       for k, v in seg.kb.params.items()]),
                         interactive=False, label="Параметры (пороги)", wrap=True, column_widths=["22%", "10%", "68%"])
            gr.Dataframe(pd.DataFrame([{
                "id": r["id"], "Правило": r["name"],
                "ЕСЛИ": " И ".join(f"{a} {o} {b}" for a, o, b in r["if"]).replace("True", "да"),
                "ТО": "; ".join(filter(None, [
                    f"уровень {r['then']['severity']} ({seg.kb.levels[r['then']['severity']]['name']})"
                    if "severity" in r["then"] else "",
                    ", ".join(f"{k} = да" if v is True else f"{k} = {v}" for k, v in r["then"].get("facts", {}).items()),
                    "" if ("severity" in r["then"] or r["then"].get("facts")) else "сообщение и рекомендация"]))}
                for r in seg.kb.rules]), interactive=False, label="Продукционные правила", wrap=True,
                max_height=900, column_widths=["6%", "22%", "44%", "28%"])
    return demo


def main():
    args = parse_args()
    seg = Segmenter(args.ckpt, args.data, args.split, args.kb, args.journal)
    if args.examples_dir:
        seg.examples = sorted(str(p.resolve()) for p in Path(args.examples_dir).iterdir()
                              if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    seg.examples = seg.examples[: args.n_examples]
    print(f"Модель загружена ({seg.meta['arch']}/{seg.meta['encoder']}, {seg.device}); "
          f"примеров: {len(seg.examples)}; журнал: {seg.journal.path}")
    launch_kw = style_kwargs() if _gradio_major() >= 6 else {}
    build_ui(seg).launch(share=args.share, server_port=args.port, show_error=True, inbrowser=args.inbrowser,
                         allowed_paths=sorted({str(Path(p).parent) for p in seg.examples} | {str(OUT_DIR)}),
                         **launch_kw)


if __name__ == "__main__":
    main()


## 1. Окружение

In [ ]:
!pip install -q segmentation-models-pytorch ultralytics
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python -m pytest -q tests

In [ ]:
DATA_ROOT = '/kaggle/input'
MASK_MODE = 'sam'
EPOCHS = 40
BATCH = 8
WORKERS = 4
MODELS = [('unet', 'resnet34'), ('deeplabv3plus', 'resnet34'), ('fpn', 'resnet34')]
RUN_BOX_ABLATION = False

## 2. Формирование набора данных
Рамки разметки преобразуются в попиксельные маски, данные делятся на train/val/test = 70/15/15.

In [ ]:
!python scripts/prepare_dataset.py --data-root {DATA_ROOT} --out prepared --mask-mode {MASK_MODE} --vis 16

In [ ]:
from IPython.display import Image, display
import glob
display(Image('prepared/class_distribution.png'))
for p in sorted(glob.glob('prepared/examples/*.jpg'))[:6]:
    display(Image(p))

## 3. Обучение моделей

In [ ]:
for arch, enc in MODELS:
    !python scripts/train.py --data prepared --arch {arch} --encoder {enc} --epochs {EPOCHS} --batch-size {BATCH} --workers {WORKERS} --out runs/{arch}_{enc}

In [ ]:
if RUN_BOX_ABLATION:
    !python scripts/prepare_dataset.py --data-root {DATA_ROOT} --out prepared_box --mask-mode box --vis 0
    !python scripts/train.py --data prepared_box --arch unet --encoder resnet34 --epochs {EPOCHS} --batch-size {BATCH} --workers {WORKERS} --out runs/unet_resnet34_boxmasks

In [ ]:
for arch, enc in MODELS:
    print(arch, enc); display(Image(f'runs/{arch}_{enc}/training_curves.png'))

## 4. Тестирование и метрики (IoU, Dice, точность, полнота)

In [ ]:
for arch, enc in MODELS:
    !python scripts/evaluate.py --ckpt runs/{arch}_{enc}/best.pt --data prepared --split test --tta
runs = ' '.join(f'runs/{a}_{e}' for a, e in MODELS)
!python scripts/compare_models.py {runs} --out runs/comparison.csv

In [ ]:
import pandas as pd
best = pd.read_csv('runs/comparison.csv').iloc[0]
print('Лучшая модель:', best['модель'])
BEST_RUN = 'runs/' + best['модель'].replace('/', '_')
display(pd.read_csv(f'{BEST_RUN}/eval_test/metrics_table.csv'))
display(Image(f'{BEST_RUN}/eval_test/confusion_matrix.png'))
display(Image(f'{BEST_RUN}/eval_test/metrics_by_class.png'))
for p in sorted(glob.glob(f'{BEST_RUN}/eval_test/vis/*.jpg'))[:8]:
    display(Image(p))

## 5. Применение к новым кадрам

In [ ]:
import pandas as pd
test_imgs = pd.read_csv('prepared/index.csv').query("split=='test'").image_path.head(10)
os.makedirs('demo_in', exist_ok=True)
import shutil
for p in test_imgs: shutil.copy(p, 'demo_in/')
!python scripts/predict.py --ckpt {BEST_RUN}/best.pt --input demo_in --out predictions
for p in sorted(glob.glob('predictions/overlay/*.jpg'))[:5]:
    display(Image(p))

## 6. Оценка состояния покрытия (база знаний + машина вывода)
Правила применяются к эталонной и к предсказанной маске каждого тестового кадра; сравниваются итоговые оценки состояния.

In [ ]:
!python scripts/evaluate_rules.py --ckpt {BEST_RUN}/best.pt --data prepared --split test

## 7. Архив результатов
Скачайте из *Output*: `results_small.zip` (метрики, графики, примеры — несколько МБ) и `best_model.pt` (веса лучшей модели).

In [ ]:
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip road_defect_seg/runs road_defect_seg/prepared/dataset_stats.json road_defect_seg/prepared/class_distribution.png road_defect_seg/prepared/examples road_defect_seg/predictions -x '*.pt' && ls -lh results_small.zip
import shutil; shutil.copy(f'{BEST_RUN}/best.pt', '/kaggle/working/best_model.pt')

## 8. Веб-интерфейс
Ячейка выводит публичную ссылку `https://….gradio.live` (действует ~72 ч, пока работает ноутбук). Откройте её: можно загрузить своё фото или выбрать кадр из тестовой выборки — рядом покажется эталонная разметка.

In [ ]:
!pip install -q gradio
from app import Segmenter, build_ui, style_kwargs, _gradio_major
seg = Segmenter(f'{BEST_RUN}/best.pt', 'prepared', 'test')
seg.examples = seg.examples[:24]
demo = build_ui(seg)
demo.launch(share=True, allowed_paths=sorted({os.path.dirname(p) for p in seg.examples}),
            **(style_kwargs() if _gradio_major() >= 6 else {}))